In [ ]:
# ============================================================
# UC10 - Feature Engineering for Claims/Pharmacy/Auth Dataset
# Run this in Google Colab
# ============================================================

# ---- 1. Install/Import ----
import pandas as pd
import numpy as np

# ---- 2. Load the dataset ----
from google.colab import files
uploaded = files.upload()
df = pd.read_excel(list(uploaded.keys())[0])

print("Loaded shape:", df.shape)

# ---- 3. Parse date columns ----
date_cols = ['Service_Date', 'Service_End_Date', 'Submission_Date',
             'Processed_Date', 'Decision_Date', 'Ingestion_Timestamp']
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# Derive a clean "Batch_Date" from Batch_ID (format: BATCH_YYYYMMDD)
df['Batch_Date'] = pd.to_datetime(
    df['Batch_ID'].str.replace('BATCH_', '', regex=False),
    format='%Y%m%d', errors='coerce'
)

# ============================================================
# FEATURE 1: Provider-level denial/rejection rate
# ============================================================
denied_like = df['Status'].isin(['DENIED', 'REJECTED'])
provider_stats = (
    df.groupby('Provider_NPI')
      .agg(Provider_Total_Records=('Record_ID', 'count'),
           Provider_Denied_Records=('Status', lambda s: s.isin(['DENIED', 'REJECTED']).sum()))
      .reset_index()
)
provider_stats['Provider_Denial_Rate'] = (
    provider_stats['Provider_Denied_Records'] / provider_stats['Provider_Total_Records']
).round(4)

df = df.merge(
    provider_stats[['Provider_NPI', 'Provider_Total_Records', 'Provider_Denial_Rate']],
    on='Provider_NPI', how='left'
)

# ============================================================
# FEATURE 2: Batch-level daily volume & SLA-breach-rate trend
# (rolling 7-day average, compared against each batch's own rate)
# ============================================================
batch_daily = (
    df.groupby('Batch_Date')
      .agg(Batch_Volume=('Record_ID', 'count'),
           Batch_SLA_Breach_Rate=('SLA_Breach_Flag', lambda s: (s == 'Y').mean()))
      .reset_index()
      .sort_values('Batch_Date')
)
batch_daily['Rolling_7D_Avg_Volume'] = (
    batch_daily['Batch_Volume'].rolling(window=7, min_periods=1).mean().round(2)
)
batch_daily['Rolling_7D_Avg_SLA_Breach_Rate'] = (
    batch_daily['Batch_SLA_Breach_Rate'].rolling(window=7, min_periods=1).mean().round(4)
)
# Volume/SLA anomaly vs. its own trailing trend
batch_daily['Volume_Vs_Trend_Ratio'] = (
    batch_daily['Batch_Volume'] / batch_daily['Rolling_7D_Avg_Volume']
).round(2)
batch_daily['SLA_Breach_Rate_Vs_Trend_Diff'] = (
    batch_daily['Batch_SLA_Breach_Rate'] - batch_daily['Rolling_7D_Avg_SLA_Breach_Rate']
).round(4)

df = df.merge(
    batch_daily[['Batch_Date', 'Batch_Volume', 'Rolling_7D_Avg_Volume',
                 'Volume_Vs_Trend_Ratio', 'Batch_SLA_Breach_Rate',
                 'Rolling_7D_Avg_SLA_Breach_Rate', 'SLA_Breach_Rate_Vs_Trend_Diff']],
    on='Batch_Date', how='left'
)

# ============================================================
# FEATURE 3: Beneficiary claim frequency (how many records per member)
# ============================================================
bene_freq = (
    df.groupby('BENE_ID')
      .size()
      .reset_index(name='Beneficiary_Record_Count')
)
df = df.merge(bene_freq, on='BENE_ID', how='left')

# Flag unusually high-frequency beneficiaries (top 1% by record count)
if df['Beneficiary_Record_Count'].notna().sum() > 0:
    freq_threshold = df['Beneficiary_Record_Count'].quantile(0.99)
    df['High_Frequency_Beneficiary_Flag'] = df['Beneficiary_Record_Count'] > freq_threshold
else:
    df['High_Frequency_Beneficiary_Flag'] = False

# ============================================================
# FEATURE 4: Claim <-> Prior-Auth linkage
# Populate Auth_Linked_ID by matching claims (that require auth) to a
# PRIOR_AUTH record for the same beneficiary within a reasonable time window,
# then flag claims that require auth but have NO matching auth record.
# ============================================================
auth_records = df[df['Record_Type'] == 'PRIOR_AUTH'][
    ['Record_ID', 'BENE_ID', 'Submission_Date', 'Status']
].rename(columns={'Record_ID': 'Matched_Auth_Record_ID',
                   'Submission_Date': 'Auth_Submission_Date',
                   'Status': 'Auth_Status'})

needs_auth = df[(df['Auth_Required_Flag'] == 'Y') & (df['Record_Type'] != 'PRIOR_AUTH')].copy()

# Nearest prior PRIOR_AUTH submission date per BENE_ID
matches = needs_auth.merge(auth_records, on='BENE_ID', how='left')
matches = matches[matches['Auth_Submission_Date'] <= matches['Submission_Date']]
matches['Days_Between'] = (matches['Submission_Date'] - matches['Auth_Submission_Date']).dt.days
matches = matches.sort_values('Days_Between').drop_duplicates(subset='Record_ID', keep='first')

link_map = matches.set_index('Record_ID')['Matched_Auth_Record_ID']
df['Auth_Linked_ID'] = df['Record_ID'].map(link_map).combine_first(df['Auth_Linked_ID'])

df['Missing_Required_Auth_Link'] = (
    (df['Auth_Required_Flag'] == 'Y') &
    (df['Record_Type'] != 'PRIOR_AUTH') &
    (df['Auth_Linked_ID'].isna())
)

# ============================================================
# FEATURE 5: Time since last batch per source system (pipeline gap detection)
# ============================================================
sys_batches = (
    df[['Source_System', 'Batch_Date']]
    .dropna()
    .drop_duplicates()
    .sort_values(['Source_System', 'Batch_Date'])
)
sys_batches['Days_Since_Prev_Batch'] = (
    sys_batches.groupby('Source_System')['Batch_Date'].diff().dt.days
)
df = df.merge(sys_batches, on=['Source_System', 'Batch_Date'], how='left')

# Flag a pipeline gap if the gap is unusually large (> 3x the median gap for that system)
gap_threshold_map = sys_batches.groupby('Source_System')['Days_Since_Prev_Batch'].median() * 3
df['Pipeline_Gap_Flag'] = df.apply(
    lambda r: (r['Days_Since_Prev_Batch'] > gap_threshold_map.get(r['Source_System'], np.inf))
    if pd.notna(r['Days_Since_Prev_Batch']) else False,
    axis=1
)

# ============================================================
# FEATURE 6: Day-of-week seasonality-normalized SLA breach rate
# ============================================================
df['Submission_Day_Of_Week'] = df['Submission_Date'].dt.day_name()

dow_breach_rate = (
    df.groupby('Submission_Day_Of_Week')['SLA_Breach_Flag']
      .apply(lambda s: (s == 'Y').mean())
      .rename('DOW_Avg_SLA_Breach_Rate')
      .reset_index()
)
df = df.merge(dow_breach_rate, on='Submission_Day_Of_Week', how='left')

df['Record_SLA_Breach_Numeric'] = (df['SLA_Breach_Flag'] == 'Y').astype(int)
df['SLA_Breach_Vs_DOW_Norm'] = (
    df['Record_SLA_Breach_Numeric'] - df['DOW_Avg_SLA_Breach_Rate']
).round(4)

# ============================================================
# 4. Save the feature-engineered dataset
# ============================================================
output_path = "claims_pharmacy_auth_monitor_dataset_features.csv"
df.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Final shape:", df.shape)
print("\nNew feature columns added:")
new_cols = ['Provider_Total_Records', 'Provider_Denial_Rate',
            'Batch_Volume', 'Rolling_7D_Avg_Volume', 'Volume_Vs_Trend_Ratio',
            'Batch_SLA_Breach_Rate', 'Rolling_7D_Avg_SLA_Breach_Rate', 'SLA_Breach_Rate_Vs_Trend_Diff',
            'Beneficiary_Record_Count', 'High_Frequency_Beneficiary_Flag',
            'Auth_Linked_ID', 'Missing_Required_Auth_Link',
            'Days_Since_Prev_Batch', 'Pipeline_Gap_Flag',
            'Submission_Day_Of_Week', 'DOW_Avg_SLA_Breach_Rate', 'SLA_Breach_Vs_DOW_Norm']
for c in new_cols:
    print(" -", c)

df.head(10)

Saving claims_pharmacy_auth_monitor_dataset_final.xlsx to claims_pharmacy_auth_monitor_dataset_final (5).xlsx
Loaded shape: (10000, 32)
Saved: claims_pharmacy_auth_monitor_dataset_features.csv
Final shape: (10000, 50)

New feature columns added:
 - Provider_Total_Records
 - Provider_Denial_Rate
 - Batch_Volume
 - Rolling_7D_Avg_Volume
 - Volume_Vs_Trend_Ratio
 - Batch_SLA_Breach_Rate
 - Rolling_7D_Avg_SLA_Breach_Rate
 - SLA_Breach_Rate_Vs_Trend_Diff
 - Beneficiary_Record_Count
 - High_Frequency_Beneficiary_Flag
 - Auth_Linked_ID
 - Missing_Required_Auth_Link
 - Days_Since_Prev_Batch
 - Pipeline_Gap_Flag
 - Submission_Day_Of_Week
 - DOW_Avg_SLA_Breach_Rate
 - SLA_Breach_Vs_DOW_Norm


,Record_ID,Record_Type,BENE_ID,Provider_NPI,Provider_State,Service_Date,Service_End_Date,Submission_Date,Processed_Date,Decision_Date,...,SLA_Breach_Rate_Vs_Trend_Diff,Beneficiary_Record_Count,High_Frequency_Beneficiary_Flag,Missing_Required_Auth_Link,Days_Since_Prev_Batch,Pipeline_Gap_Flag,Submission_Day_Of_Week,DOW_Avg_SLA_Breach_Rate,Record_SLA_Breach_Numeric,SLA_Breach_Vs_DOW_Norm
0,PH202054,PHARMACY_CLAIM,-1.000001e+13,1.196713e+09,AZ,2025-07-24,2025-07-24,2025-07-24,2025-07-24 00:00:00,NaT,...,0.0944,2.0,False,True,1.0,False,Thursday,0.159190,0,-0.1592
1,MC100442,MEDICAL_CLAIM,-1.000001e+13,1.811172e+09,DE,2017-10-05,2017-10-05,2017-10-12,2017-10-06 00:00:00,NaT,...,0.0000,2.0,False,False,1.0,False,Thursday,0.159190,0,-0.1592
2,MC103954,MEDICAL_CLAIM,-1.000001e+13,1.073193e+09,GA,2018-12-26,2018-12-26,2018-12-31,2018-12-28 00:00:00,NaT,...,0.0000,2.0,False,False,1.0,False,Monday,0.149931,0,-0.1499
3,MC102288,MEDICAL_CLAIM,-1.000001e+13,1.689689e+09,CO,2016-01-26,2016-01-26,2016-01-30,2016-01-29 00:00:00,NaT,...,0.0000,29.0,True,False,1.0,False,Saturday,0.150972,0,-0.1510
4,MC103196,MEDICAL_CLAIM,-1.000001e+13,1.194177e+09,MD,2019-10-10,2019-10-10,2019-10-14,2019-10-11 00:00:00,NaT,...,0.0000,4.0,False,False,2.0,False,Monday,0.149931,0,-0.1499
5,PH201178,PHARMACY_CLAIM,-1.000001e+13,1.575306e+09,AZ,2025-11-12,2025-11-12,2025-11-12,2025-11-15 00:00:00,NaT,...,-0.0105,4.0,False,False,1.0,False,Wednesday,0.158741,1,0.8413
6,PA300351,PRIOR_AUTH,-1.000001e+13,1.841233e+09,AL,NaT,NaT,2025-04-19,2025-04-24 12:00:00,2025-04-24 12:00:00,...,-0.1389,2.0,False,False,1.0,False,Saturday,0.150972,0,-0.1510
7,PH200658,PHARMACY_CLAIM,NaN,1.204276e+09,CA,2025-10-14,2025-10-14,2025-10-14,2025-10-16 00:00:00,NaT,...,0.1999,NaN,False,False,1.0,False,Tuesday,0.167529,0,-0.1675
8,MC102065,MEDICAL_CLAIM,-1.000001e+13,1.740514e+09,FL,2022-08-29,2022-08-29,2022-09-04,2022-09-02 00:00:00,NaT,...,0.0000,3.0,False,False,1.0,False,Sunday,0.152159,0,-0.1522
9,MC100413,MEDICAL_CLAIM,-1.000001e+13,1.700238e+09,GA,2020-02-20,2020-02-20,2020-02-21,2020-02-21 00:00:00,NaT,...,0.0000,10.0,False,False,1.0,False,Friday,0.171190,0,-0.1712


In [ ]:
# ============================================================
# UC10 - Data Quality Engine (Profiler + Rule Checks + Scoring)
# Runs AFTER the feature-engineering step.
# Input : claims_pharmacy_auth_monitor_dataset_features.csv
# Output: outputs/data_profile.json, outputs/quality_report.json,
#         outputs/batch_sla_risk.json
# Run this in Google Colab
# ============================================================

import pandas as pd
import numpy as np
import json
import os
import datetime

# ============================================================
# STEP 0: Load the feature-engineered dataset
# ============================================================
DATA_PATH = "claims_pharmacy_auth_monitor_dataset_features.csv"   # <-- output of the feature engineering script
OUTPUT_DIR = "outputs"

dtype_spec = {
    "Record_ID": str,
    "BENE_ID": str,
    "Provider_NPI": str,
    "Auth_Linked_ID": str
}

print(f"Loading {DATA_PATH} ...")
df = pd.read_csv(DATA_PATH, dtype=dtype_spec)

date_columns = ["Service_Date", "Service_End_Date", "Processed_Date", "Decision_Date", "Submission_Date"]
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"Loaded {len(df):,} rows, {len(df.columns)} columns.")


# ============================================================
# STEP 1: PROFILER
# Produces a structural snapshot of the dataset - per-column
# missingness, uniqueness, ranges, and top values. This is the
# "what does the raw data look like" layer, before any rule judges it.
# ============================================================
def generate_profile(df, output_path=f"{OUTPUT_DIR}/data_profile.json"):
    profile = {}

    profile["total_records"] = len(df)
    profile["total_columns"] = len(df.columns)
    profile["columns"] = {}

    # exact duplicate rows (every column identical)
    profile["exact_duplicate_rows"] = int(df.duplicated().sum())

    # duplicate Record_ID values (same ID appearing more than once)
    if "Record_ID" in df.columns:
        profile["duplicate_record_ids"] = int(df["Record_ID"].duplicated().sum())
    else:
        profile["duplicate_record_ids"] = 0

    for col in df.columns:
        col_data = df[col]
        col_profile = {}

        col_profile["data_type"] = str(col_data.dtype)

        missing_count = int(col_data.isnull().sum())
        col_profile["missing_count"] = missing_count
        col_profile["missing_percentage"] = float(missing_count / len(df) * 100) if len(df) > 0 else 0.0

        col_profile["unique_count"] = int(col_data.nunique(dropna=True))

        if pd.api.types.is_numeric_dtype(col_data):
            col_profile["min"] = float(col_data.min()) if pd.notnull(col_data.min()) else None
            col_profile["max"] = float(col_data.max()) if pd.notnull(col_data.max()) else None
            col_profile["mean"] = float(col_data.mean()) if pd.notnull(col_data.mean()) else None

        elif pd.api.types.is_object_dtype(col_data) or isinstance(col_data.dtype, pd.CategoricalDtype):
            col_profile["value_counts"] = col_data.value_counts(dropna=True).head(10).to_dict()

        profile["columns"][col] = col_profile

    profile["date_parsing_failures"] = {}

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(profile, f, indent=4)

    return profile


# ============================================================
# STEP 2: RULE CATALOG
# 21 rules across 5 data-quality dimensions: Completeness,
# Validity, Uniqueness, Consistency, Timeliness. Each rule
# carries a severity, the fields it checks, a plain-language
# description, and a recommended fix - so every failure is
# immediately explainable, not just a flagged number.
# ============================================================
RULES = [
    {"rule_id": "R001", "rule_name": "Record_ID Completeness", "dimension": "Completeness", "severity": "Critical",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"], "fields": ["Record_ID"],
     "description": "Record_ID must not be missing.",
     "recommended_fix": "Investigate source system extraction logic. All records must have a primary identifier."},

    {"rule_id": "R002", "rule_name": "Record_ID Uniqueness", "dimension": "Uniqueness", "severity": "Critical",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"], "fields": ["Record_ID"],
     "description": "Record_ID must be unique.",
     "recommended_fix": "Deduplicate records based on Record_ID or check if upstream systems are sending multiple updates as new records."},

    {"rule_id": "R003", "rule_name": "Beneficiary and Provider Completeness", "dimension": "Completeness", "severity": "High",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"], "fields": ["BENE_ID", "Provider_NPI"],
     "description": "BENE_ID and Provider_NPI must be present for every record type.",
     "recommended_fix": "Ensure patient and provider contexts are fully mapped in the data pipeline."},

    {"rule_id": "R004", "rule_name": "Provider_NPI Validity", "dimension": "Validity", "severity": "High",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"], "fields": ["Provider_NPI"],
     "description": "Provider_NPI must contain exactly 10 digits when present.",
     "recommended_fix": "Validate NPI format against the National Plan and Provider Enumeration System standard."},

    {"rule_id": "R005", "rule_name": "Record_ID Prefix Consistency", "dimension": "Consistency", "severity": "High",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"], "fields": ["Record_ID", "Record_Type"],
     "description": "Record_ID prefix must match Record_Type (MC, PH, PA).",
     "recommended_fix": "Check for ID generation errors or mismatched Record_Type assignments."},

    {"rule_id": "R006", "rule_name": "Source System Consistency", "dimension": "Consistency", "severity": "High",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"], "fields": ["Record_Type", "Source_System"],
     "description": "Record_Type must map to the correct Source_System.",
     "recommended_fix": "Correct source system mapping tables in the ETL logic."},

    {"rule_id": "R007", "rule_name": "Medical Claim Core Fields Completeness", "dimension": "Completeness", "severity": "High",
     "applicable_record_types": ["MEDICAL_CLAIM"],
     "fields": ["Service_Date", "Service_End_Date", "Diagnosis_Code", "Billed_Amount", "Allowed_Amount", "Paid_Amount", "Patient_Responsibility"],
     "description": "MEDICAL_CLAIM requires specific dates, codes, and financial amounts.",
     "recommended_fix": "Verify that all required medical claim fields are extracted from CARRIER_CLAIMS_SYS."},

    {"rule_id": "R008", "rule_name": "Pharmacy Claim Core Fields Completeness", "dimension": "Completeness", "severity": "High",
     "applicable_record_types": ["PHARMACY_CLAIM"],
     "fields": ["Service_Date", "Service_End_Date", "NDC_Code", "Drug_Name", "Days_Supply", "Quantity_Dispensed", "Billed_Amount", "Allowed_Amount", "Paid_Amount", "Patient_Responsibility"],
     "description": "PHARMACY_CLAIM requires specific dates, drug details, and financial amounts.",
     "recommended_fix": "Verify that all required pharmacy claim fields are extracted from PHARMACY_ADJ_SYS."},

    {"rule_id": "R009", "rule_name": "Prior Auth Core Fields Completeness", "dimension": "Completeness", "severity": "High",
     "applicable_record_types": ["PRIOR_AUTH"], "fields": ["Procedure_Code", "Urgency_Flag", "Processed_Date", "Decision_Date", "Status"],
     "description": "PRIOR_AUTH requires codes and urgency, plus dates if APPROVED or DENIED.",
     "recommended_fix": "Ensure decision dates are captured when prior authorizations leave the PENDING state."},

    {"rule_id": "R010", "rule_name": "Financial Amount Non-Negative Validity", "dimension": "Validity", "severity": "High",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM"], "fields": ["Billed_Amount", "Allowed_Amount", "Paid_Amount", "Patient_Responsibility"],
     "description": "Financial amounts for claims must not be negative.",
     "recommended_fix": "Review adjustment or reversal logic to ensure final line amounts are non-negative."},

    {"rule_id": "R011", "rule_name": "Pharmacy Supply Validity", "dimension": "Validity", "severity": "High",
     "applicable_record_types": ["PHARMACY_CLAIM"], "fields": ["Days_Supply", "Quantity_Dispensed"],
     "description": "Days_Supply and Quantity_Dispensed must be greater than zero.",
     "recommended_fix": "Investigate pharmacy dispensing data for zero values."},

    {"rule_id": "R012", "rule_name": "Status Validity by Record Type", "dimension": "Validity", "severity": "High",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"], "fields": ["Record_Type", "Status"],
     "description": "Status must be a valid value for the given Record_Type.",
     "recommended_fix": "Update allowed value lists or correct status mapping during data ingestion."},

    {"rule_id": "R013", "rule_name": "Denial Reason Completeness", "dimension": "Completeness", "severity": "Medium",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"], "fields": ["Status", "Denial_Reason_Code"],
     "description": "DENIED or REJECTED records must have a Denial_Reason_Code.",
     "recommended_fix": "Ensure denial codes are populated whenever a claim or auth is denied."},

    {"rule_id": "R014", "rule_name": "Service Dates Consistency", "dimension": "Consistency", "severity": "High",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM"], "fields": ["Service_Date", "Service_End_Date"],
     "description": "Service_End_Date must be on or after Service_Date.",
     "recommended_fix": "Fix date entry errors where end date precedes start date."},

    {"rule_id": "R015", "rule_name": "Submission vs Service Date Consistency", "dimension": "Consistency", "severity": "High",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"], "fields": ["Service_Date", "Submission_Date"],
     "description": "Submission_Date must not be before Service_Date.",
     "recommended_fix": "Investigate time zone issues or data entry errors causing submissions prior to service."},

    {"rule_id": "R016", "rule_name": "Processed vs Submission Date Consistency", "dimension": "Consistency", "severity": "Critical",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"], "fields": ["Submission_Date", "Processed_Date"],
     "description": "Processed_Date must not be before Submission_Date.",
     "recommended_fix": "Investigate system clock sync or ETL latency causing processed date anomalies."},

    {"rule_id": "R017", "rule_name": "Auth Link Consistency", "dimension": "Consistency", "severity": "Critical",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM"], "fields": ["Auth_Required_Flag", "Auth_Linked_ID"],
     "description": "If Auth_Required_Flag is Y, Auth_Linked_ID must exist.",
     "recommended_fix": "Ensure auth numbers are carried over into claims processing systems."},

    {"rule_id": "R018", "rule_name": "Auth Link Referential Integrity", "dimension": "Consistency", "severity": "High",
     "applicable_record_types": ["MEDICAL_CLAIM", "PHARMACY_CLAIM"], "fields": ["Auth_Linked_ID"],
     "description": "If Auth_Linked_ID exists, it must match an existing PRIOR_AUTH record.",
     "recommended_fix": "Check for orphaned claims where the prior authorization is missing from the dataset."},
]
# Note: SLA/Timeliness rules (SLA breach flag accuracy, volume trend consistency,
# SLA breach rate trend consistency) are intentionally excluded from this catalog.
# They will run as part of a separate SLA risk pipeline downstream of this
# data-quality check.


# ============================================================
# STEP 3: QUALITY ENGINE
# Executes every rule from the catalog against the records it
# applies to (by Record_Type), and records a pass/fail outcome
# with the failure rate and sample offending Record_IDs.
# ============================================================
def run_quality_checks(df, rules):
    results = []

    for rule in rules:
        rule_id = rule["rule_id"]
        applicable_types = rule["applicable_record_types"]
        mask = df["Record_Type"].isin(applicable_types)
        applicable_df = df[mask]

        total_applicable = len(applicable_df)
        if total_applicable == 0:
            continue

        failed_mask = pd.Series(False, index=applicable_df.index)

        if rule_id == "R001":
            failed_mask = applicable_df["Record_ID"].isnull() | (applicable_df["Record_ID"] == "")

        elif rule_id == "R002":
            failed_mask = applicable_df.duplicated(subset=["Record_ID"], keep=False)

        elif rule_id == "R003":
            failed_mask = (applicable_df["BENE_ID"].isnull() | applicable_df["Provider_NPI"].isnull()
                            | (applicable_df["BENE_ID"] == "") | (applicable_df["Provider_NPI"] == ""))

        elif rule_id == "R004":
            failed_mask = (applicable_df["Provider_NPI"].notnull() & (applicable_df["Provider_NPI"] != "")
                            & (applicable_df["Provider_NPI"].astype(str).str.len() != 10))

        elif rule_id == "R005":
            mc_fail = (applicable_df["Record_Type"] == "MEDICAL_CLAIM") & ~applicable_df["Record_ID"].astype(str).str.startswith("MC")
            ph_fail = (applicable_df["Record_Type"] == "PHARMACY_CLAIM") & ~applicable_df["Record_ID"].astype(str).str.startswith("PH")
            pa_fail = (applicable_df["Record_Type"] == "PRIOR_AUTH") & ~applicable_df["Record_ID"].astype(str).str.startswith("PA")
            failed_mask = mc_fail | ph_fail | pa_fail

        elif rule_id == "R006":
            mc_fail = (applicable_df["Record_Type"] == "MEDICAL_CLAIM") & (applicable_df["Source_System"] != "CARRIER_CLAIMS_SYS")
            ph_fail = (applicable_df["Record_Type"] == "PHARMACY_CLAIM") & (applicable_df["Source_System"] != "PHARMACY_ADJ_SYS")
            pa_fail = (applicable_df["Record_Type"] == "PRIOR_AUTH") & (applicable_df["Source_System"] != "AUTH_MGMT_SYS")
            failed_mask = mc_fail | ph_fail | pa_fail

        elif rule_id == "R007":
            cols = ["Service_Date", "Service_End_Date", "Diagnosis_Code", "Billed_Amount", "Allowed_Amount", "Paid_Amount", "Patient_Responsibility"]
            failed_mask = applicable_df[cols].isnull().any(axis=1)

        elif rule_id == "R008":
            cols = ["Service_Date", "Service_End_Date", "NDC_Code", "Drug_Name", "Days_Supply", "Quantity_Dispensed", "Billed_Amount", "Allowed_Amount", "Paid_Amount", "Patient_Responsibility"]
            failed_mask = applicable_df[cols].isnull().any(axis=1)

        elif rule_id == "R009":
            missing_base = applicable_df[["Procedure_Code", "Urgency_Flag"]].isnull().any(axis=1)
            needs_dates = applicable_df["Status"].isin(["APPROVED", "DENIED"])
            missing_dates = needs_dates & applicable_df[["Processed_Date", "Decision_Date"]].isnull().any(axis=1)
            failed_mask = missing_base | missing_dates

        elif rule_id == "R010":
            cols = ["Billed_Amount", "Allowed_Amount", "Paid_Amount", "Patient_Responsibility"]
            failed_mask = (applicable_df[cols] < 0).any(axis=1)

        elif rule_id == "R011":
            cols = ["Days_Supply", "Quantity_Dispensed"]
            failed_mask = (applicable_df[cols] <= 0).any(axis=1)

        elif rule_id == "R012":
            mc_fail = (applicable_df["Record_Type"] == "MEDICAL_CLAIM") & ~applicable_df["Status"].isin(["PAID", "DENIED", "PENDING"])
            ph_fail = (applicable_df["Record_Type"] == "PHARMACY_CLAIM") & ~applicable_df["Status"].isin(["PAID", "REJECTED", "PENDING"])
            pa_fail = (applicable_df["Record_Type"] == "PRIOR_AUTH") & ~applicable_df["Status"].isin(["APPROVED", "DENIED", "PENDING"])
            failed_mask = mc_fail | ph_fail | pa_fail

        elif rule_id == "R013":
            needs_reason = applicable_df["Status"].isin(["DENIED", "REJECTED"])
            missing_reason = applicable_df["Denial_Reason_Code"].isnull() | (applicable_df["Denial_Reason_Code"] == "")
            failed_mask = needs_reason & missing_reason

        elif rule_id == "R014":
            has_dates = applicable_df["Service_Date"].notnull() & applicable_df["Service_End_Date"].notnull()
            failed_mask = has_dates & (applicable_df["Service_End_Date"] < applicable_df["Service_Date"])

        elif rule_id == "R015":
            has_dates = applicable_df["Service_Date"].notnull() & applicable_df["Submission_Date"].notnull()
            failed_mask = has_dates & (applicable_df["Submission_Date"] < applicable_df["Service_Date"])

        elif rule_id == "R016":
            has_dates = applicable_df["Submission_Date"].notnull() & applicable_df["Processed_Date"].notnull()
            failed_mask = has_dates & (applicable_df["Processed_Date"] < applicable_df["Submission_Date"])

        elif rule_id == "R017":
            auth_req = applicable_df["Auth_Required_Flag"] == "Y"
            auth_missing = applicable_df["Auth_Linked_ID"].isnull() | (applicable_df["Auth_Linked_ID"] == "")
            failed_mask = auth_req & auth_missing

        elif rule_id == "R018":
            has_auth = applicable_df["Auth_Linked_ID"].notnull() & (applicable_df["Auth_Linked_ID"] != "")
            valid_auths = df[df["Record_Type"] == "PRIOR_AUTH"]["Record_ID"].unique()
            failed_mask = has_auth & ~applicable_df["Auth_Linked_ID"].isin(valid_auths)

        affected_count = int(failed_mask.sum())
        failure_rate = (affected_count / total_applicable) * 100 if total_applicable > 0 else 0

        status = "PASSED" if affected_count == 0 else "FAILED"
        sample_ids = applicable_df[failed_mask]["Record_ID"].head(10).tolist() if "Record_ID" in applicable_df.columns else []

        results.append({
            "rule_id": rule_id,
            "rule_name": rule["rule_name"],
            "dimension": rule["dimension"],
            "severity": rule["severity"],
            "status": status,
            "total_applicable_records": total_applicable,
            "affected_records": affected_count,
            "failure_rate_pct": failure_rate,
            "sample_record_ids": sample_ids,
            "fields": rule["fields"],
            "message": f"{affected_count} records failed the rule." if affected_count > 0 else "All records passed.",
            "description": rule["description"],
            "recommended_fix": rule["recommended_fix"]
        })

    return results


# ============================================================
# STEP 4: SCORING
# Rolls up rule results into 4 dimension scores (0-100) and a
# weighted overall quality score. SLA/Timeliness is intentionally
# excluded here - it runs as its own downstream SLA risk pipeline,
# not as part of this data-quality check.
# ============================================================
def calculate_scores_and_risk(rule_results, df, output_dir=OUTPUT_DIR):
    dimensions = ["Completeness", "Validity", "Uniqueness", "Consistency"]
    dim_scores = {dim: 100.0 for dim in dimensions}
    dim_rates = {dim: [] for dim in dimensions}
    critical_failures = 0
    top_failed_rules = []

    for res in rule_results:
        dim = res["dimension"]
        if dim not in dim_rates:
            dim = "Consistency"  # fallback bucket

        dim_rates[dim].append(res["failure_rate_pct"])

        if res["severity"] == "Critical" and res["status"] == "FAILED":
            critical_failures += 1

        if res["status"] == "FAILED":
            top_failed_rules.append(res)

    for dim, rates in dim_rates.items():
        if rates:
            avg_rate = sum(rates) / len(rates)
            dim_scores[dim] = max(0.0, 100.0 - avg_rate)

    # Weights rebalanced after removing Timeliness (originally 25/25/20/20/10):
    # each remaining weight is divided by 0.90 so they still sum to 1.0
    overall_score = (
        0.2778 * dim_scores.get("Completeness", 100) +
        0.2778 * dim_scores.get("Validity", 100) +
        0.2222 * dim_scores.get("Uniqueness", 100) +
        0.2222 * dim_scores.get("Consistency", 100)
    )

    if overall_score >= 90 and critical_failures == 0:
        overall_risk_level = "LOW"
    elif overall_score >= 75:
        overall_risk_level = "MEDIUM"
    elif overall_score >= 50:
        overall_risk_level = "HIGH"
    else:
        overall_risk_level = "CRITICAL"

    top_failed_rules.sort(key=lambda x: x["failure_rate_pct"], reverse=True)

    quality_report = {
        "run_timestamp": datetime.datetime.now().isoformat(),
        "records_scanned": len(df),
        "dimension_scores": dim_scores,
        "overall_quality_score": overall_score,
        "overall_risk_level": overall_risk_level,
        "critical_issue_count": critical_failures,
        "all_rule_results": rule_results,
        "top_failed_rules": top_failed_rules[:5],
    }

    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/quality_report.json", "w") as f:
        json.dump(quality_report, f, indent=4)

    return quality_report


# ============================================================
# STEP 5: RUN THE FULL PIPELINE
# ============================================================
print("Generating data profile...")
profile = generate_profile(df)
print(f"  -> outputs/data_profile.json")

print("Running quality checks (18 rules)...")
results = run_quality_checks(df, RULES)
print(f"  -> {len(results)} rules executed")

print("Calculating scores and risk...")
report = calculate_scores_and_risk(results, df)
print(f"  -> outputs/quality_report.json")

print()
print(f"Overall Quality Score : {report['overall_quality_score']:.2f} / 100")
print(f"Overall Risk Level    : {report['overall_risk_level']}")
print(f"Critical Issues       : {report['critical_issue_count']}")
print()
print("Dimension scores:")
for dim, score in report["dimension_scores"].items():
    print(f"  {dim:<15} {score:.2f}%")
print()
print("Top failed rules:")
for r in report["top_failed_rules"]:
    print(f"  {r['rule_id']} - {r['rule_name']} ({r['severity']}): {r['failure_rate_pct']:.2f}% failed")

print("\nRun successful!")

# In Colab, download the results:
# from google.colab import files
# files.download(f"{OUTPUT_DIR}/data_profile.json")
# files.download(f"{OUTPUT_DIR}/quality_report.json")



In [ ]:
# ============================================================
# UC10 - Root-Cause Classification Layer (percentages only)
# Adds failure-magnitude classification on top of the existing
# quality_report.json. SLA is a separate downstream pipeline,
# so no SLA data is referenced here.
# Run this in Google Colab AFTER the data quality engine cell.
# ============================================================

import json

with open("outputs/quality_report.json") as f:
    report = json.load(f)

# ============================================================
# Classify each rule by failure magnitude.
# A rule failing on most of the dataset almost always signals a
# systemic/structural defect (schema, casting, upstream mapping) -
# not scattered individual record errors.
# ============================================================
def classify_failure(failure_rate_pct):
    if failure_rate_pct >= 50:
        return "SYSTEMIC"
    elif failure_rate_pct >= 10:
        return "WIDESPREAD"
    elif failure_rate_pct > 0:
        return "ISOLATED"
    else:
        return "NONE"

for r in report["all_rule_results"]:
    r["failure_classification"] = classify_failure(r["failure_rate_pct"])

for r in report["top_failed_rules"]:
    r["failure_classification"] = classify_failure(r["failure_rate_pct"])

systemic = [r for r in report["all_rule_results"] if r["failure_classification"] == "SYSTEMIC"]
widespread = [r for r in report["all_rule_results"] if r["failure_classification"] == "WIDESPREAD"]
isolated = [r for r in report["all_rule_results"] if r["failure_classification"] == "ISOLATED"]
passed = [r for r in report["all_rule_results"] if r["failure_classification"] == "NONE"]

report["classification_summary"] = {
    "systemic_count": len(systemic),
    "widespread_count": len(widespread),
    "isolated_count": len(isolated),
    "passed_count": len(passed),
    "systemic_pct": round(100 * len(systemic) / len(report["all_rule_results"]), 2),
    "widespread_pct": round(100 * len(widespread) / len(report["all_rule_results"]), 2),
    "isolated_pct": round(100 * len(isolated) / len(report["all_rule_results"]), 2),
    "passed_pct": round(100 * len(passed) / len(report["all_rule_results"]), 2),
}

with open("outputs/quality_report.json", "w") as f:
    json.dump(report, f, indent=4)

print(f"Overall Quality Score : {report['overall_quality_score']:.2f}%")
print(f"Overall Risk Level    : {report['overall_risk_level']}")
print()
print("Dimension scores:")
for dim, score in report["dimension_scores"].items():
    print(f"  {dim:<15} {score:.2f}%")
print()
print("Rule classification breakdown:")
cs = report["classification_summary"]
print(f"  SYSTEMIC   : {cs['systemic_count']} rules ({cs['systemic_pct']}%)")
print(f"  WIDESPREAD : {cs['widespread_count']} rules ({cs['widespread_pct']}%)")
print(f"  ISOLATED   : {cs['isolated_count']} rules ({cs['isolated_pct']}%)")
print(f"  PASSED     : {cs['passed_count']} rules ({cs['passed_pct']}%)")
print()
print("Per-rule failure rate + classification:")
for r in sorted(report["all_rule_results"], key=lambda x: x["failure_rate_pct"], reverse=True):
    print(f"  [{r['failure_classification']:<10}] {r['rule_id']} {r['rule_name']:<45} {r['failure_rate_pct']:.2f}%")

print("\nUpdated outputs/quality_report.json with failure_classification + classification_summary.")

In [ ]:

from typing import Any
# ---------------------------------------------------------------------------
# Default configuration
# ---------------------------------------------------------------------------

_DEFAULTS: dict[str, Any] = {
    # -----------------------------------------------------------------------
    # Historical baseline
    # -----------------------------------------------------------------------
    # Rolling window size (number of past batch observations used to compute
    # the historical median and robust scale at each timepoint).
    # Rationale: 30 batches ≈ ~1 month of daily batches; gives stable baseline
    # without being overly sensitive to ancient history.
    "baseline_window": 30,

    # Minimum number of past observations required before a baseline is
    # considered reliable. Below this, baseline is set to None / NaN and
    # EWMA/CUSUM signals are suppressed.
    "baseline_min_obs": 5,

    # Configurable scale floor used when the entire historical window is constant
    # (MAD == 0 and sample std == 0). Prevents division-by-zero or infinite
    # sensitivity on numerical noise without masking genuine shifts.
    "min_scale_floor": 0.01,

    # -----------------------------------------------------------------------
    # EWMA — Exponentially Weighted Moving Average
    # -----------------------------------------------------------------------
    # Smoothing factor (0 < alpha ≤ 1).
    # alpha=0.3 gives moderate smoothing; EWMA responds to a gradual shift
    # over roughly 1/alpha ≈ 3–4 periods.
    "ewma_alpha": 0.3,

    # Number of initial observations used to seed the EWMA (warm-up).
    # The EWMA is initialised to the median of the first ewma_warmup_n values.
    "ewma_warmup_n": 5,

    # EWMA signal thresholds expressed as multiples of robust sigma.
    # These are statistical detection thresholds, NOT business SLA targets.
    "ewma_warning_sigma": 1.5,   # WARNING:  deviation ≥ 1.5 × robust_sigma
    "ewma_alert_sigma":   2.0,   # ALERT:    deviation ≥ 2.0 × robust_sigma

    # -----------------------------------------------------------------------
    # CUSUM — Cumulative Sum Control Chart
    # -----------------------------------------------------------------------
    # Reference value k (half the allowable shift, in units of robust sigma).
    # Standard choice for detecting a 1-sigma shift.
    "cusum_k_sigma": 0.5,

    # Decision threshold h (in units of robust sigma).
    # Standard value; requires a sustained cumulative deviation before signal.
    "cusum_h_sigma": 5.0,

    # -----------------------------------------------------------------------
    # Pipeline signal classification
    # Thresholds derived from observable data distribution:
    #   - Days_Since_Prev_Batch is 1.0 for 91 % of records; ≥ 2 is abnormal.
    #   - Retry_Count max is 3; ≥ 2 indicates repeated failure.
    #   - Pipeline_Gap_Flag is a boolean flag defined in the dataset.
    # No pipeline SLA target exists → no pipeline SLA breach is declared.
    # -----------------------------------------------------------------------
    "pipeline_gap_days_threshold": 2,    # Days_Since_Prev_Batch ≥ this → DEGRADED
    "pipeline_retry_threshold":    2,    # Retry_Count ≥ this (per record) → DEGRADED

    # -----------------------------------------------------------------------
    # Output
    # -----------------------------------------------------------------------
    "output_dir":               "outputs",
    "findings_filename":        "sla_temporal_findings.json",
}


def get_config(overrides: dict[str, Any] | None = None) -> dict[str, Any]:
    """Return a merged configuration dictionary.

    Parameters
    ----------
    overrides:
        Optional key/value pairs that override specific defaults.

    Returns
    -------
    dict
        Full configuration with overrides applied.
    """
    cfg = dict(_DEFAULTS)
    if overrides:
        cfg.update(overrides)
    return cfg


"""
sla_metrics.py
==============
SLA metric extraction from the feature dataframe.

Responsibilities
----------------
* Assign each record its ``sla_group`` (business population used for baseline
  segmentation and SLA assessment).
* Classify each record's temporal data validity.
* Derive per-record timeliness metrics (latency, utilisation, remaining time).
* Extract batch-level volume series (centered on Volume_Vs_Trend_Ratio).
* Extract whole-batch and group-specific breach rate time series.
* Extract batch-level pipeline observable signals.
"""


import logging
from typing import Any

import pandas as pd
import numpy as np

logger = logging.getLogger(__name__)

# Sentinel used when SLA assessment is not possible
NOT_ASSESSABLE: str = "NOT_ASSESSABLE"


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def assign_sla_groups(df: pd.DataFrame) -> pd.Series:
    """Return a Series of sla_group labels aligned with ``df``."""
    groups = df["Record_Type"].copy().astype(str)

    pa_mask = df["Record_Type"] == "PRIOR_AUTH"
    if pa_mask.any():
        urgency = df.loc[pa_mask, "Urgency_Flag"].fillna("UNKNOWN")
        groups.loc[pa_mask] = "PRIOR_AUTH_" + urgency.astype(str)

    return groups


def classify_temporal_validity(df: pd.DataFrame) -> pd.Series:
    """Classify each record's temporal data validity for SLA assessment."""
    lat = df["Processing_Latency_Days"]
    proc_date_null = df["Processed_Date"].isnull()

    validity = pd.Series("VALID", index=df.index, dtype=str)

    # Missing Processed_Date -> assessment impossible
    validity[proc_date_null] = "NULL_NO_DATE"

    # Processing_Latency_Days null while Processed_Date exists
    lat_null_with_date = lat.isnull() & ~proc_date_null
    validity[lat_null_with_date] = "MISSING_LATENCY"

    # Negative latency: Processed_Date < Service_Date (data quality problem)
    lat_negative = lat.notna() & (lat < 0)
    validity[lat_negative] = "NEGATIVE"

    return validity


def extract_record_timeliness_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """Derive per-record timeliness metric columns."""
    out = df.copy()
    out["sla_group"] = assign_sla_groups(df)
    out["temporal_validity"] = classify_temporal_validity(df)

    valid_mask = out["temporal_validity"] == "VALID"
    positive_target = out["SLA_Target_Days"].fillna(0) > 0

    # SLA utilisation: fraction of SLA budget consumed
    out["sla_utilization"] = np.nan
    assessable = valid_mask & positive_target
    out.loc[assessable, "sla_utilization"] = (
        out.loc[assessable, "Processing_Latency_Days"]
        / out.loc[assessable, "SLA_Target_Days"]
    )

    # Remaining SLA time
    out["remaining_sla_days"] = np.nan
    out.loc[assessable, "remaining_sla_days"] = (
        out.loc[assessable, "SLA_Target_Days"]
        - out.loc[assessable, "Processing_Latency_Days"]
    )

    return out


def build_batch_volume_series(df: pd.DataFrame) -> pd.DataFrame:
    """Build a chronological batch-level volume time-series."""
    batch_cols = [
        "Batch_ID", "Batch_Date",
        "Batch_Volume", "Rolling_7D_Avg_Volume", "Volume_Vs_Trend_Ratio",
    ]
    available = [c for c in batch_cols if c in df.columns]
    if "Batch_ID" not in available or "Batch_Date" not in available:
        logger.warning("Batch_ID or Batch_Date missing; volume series unavailable.")
        return pd.DataFrame()

    batch_ts = (
        df[available]
        .drop_duplicates(subset=["Batch_ID"])
        .copy()
    )
    batch_ts["Batch_Date"] = pd.to_datetime(batch_ts["Batch_Date"], errors="coerce")
    batch_ts = batch_ts.sort_values("Batch_Date").reset_index(drop=True)

    batch_ts = batch_ts.rename(columns={
        "Batch_ID":              "batch_id",
        "Batch_Date":            "batch_date",
        "Batch_Volume":          "actual_volume",
        "Rolling_7D_Avg_Volume": "baseline_volume",
        "Volume_Vs_Trend_Ratio": "volume_ratio",
    })

    if "volume_ratio" in batch_ts.columns:
        batch_ts["volume_deviation"] = batch_ts["volume_ratio"] - 1.0

    return batch_ts


def build_batch_breach_rate_series(
    df: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    """Build chronological batch-level SLA breach-rate time-series."""
    df_work = df.copy()
    df_work["sla_group"] = assign_sla_groups(df_work)
    df_work["temporal_validity"] = classify_temporal_validity(df_work)
    df_work["Batch_Date"] = pd.to_datetime(df_work["Batch_Date"], errors="coerce")

    result: dict[str, pd.DataFrame] = {}

    # 1. Whole-batch time series
    whole_batch = (
        df_work.groupby(["Batch_ID", "Batch_Date"])
        .agg(
            batch_breach_rate=("Batch_SLA_Breach_Rate", "first"),
            rolling_7d_avg_breach_rate=("Rolling_7D_Avg_SLA_Breach_Rate", "first"),
            breach_rate_vs_trend_diff=("SLA_Breach_Rate_Vs_Trend_Diff", "first"),
        )
        .reset_index()
        .sort_values("Batch_Date")
        .reset_index(drop=True)
    )
    whole_batch = whole_batch.rename(columns={"Batch_ID": "batch_id", "Batch_Date": "batch_date"})
    whole_batch["sla_group"] = "WHOLE_BATCH"
    result["WHOLE_BATCH"] = whole_batch

    # 2. Group-specific time series (derived from assessable records)
    for group in df_work["sla_group"].unique():
        grp_df = df_work[df_work["sla_group"] == group].copy()
        grp_assessable = grp_df[grp_df["temporal_validity"] == "VALID"]

        counts = grp_df.groupby(["Batch_ID", "Batch_Date"]).size().reset_index(name="total_in_grp")
        assessable_counts = grp_assessable.groupby(["Batch_ID", "Batch_Date"]).size().reset_index(name="assessable_in_grp")

        breached_mask = grp_assessable["Processing_Latency_Days"] > grp_assessable["SLA_Target_Days"]
        breached_counts = grp_assessable[breached_mask].groupby(["Batch_ID", "Batch_Date"]).size().reset_index(name="breached_in_grp")

        batch_grp = counts.merge(assessable_counts, on=["Batch_ID", "Batch_Date"], how="left").merge(breached_counts, on=["Batch_ID", "Batch_Date"], how="left")
        batch_grp["assessable_in_grp"] = batch_grp["assessable_in_grp"].fillna(0)
        batch_grp["breached_in_grp"] = batch_grp["breached_in_grp"].fillna(0)

        batch_grp["batch_breach_rate"] = np.where(
            batch_grp["assessable_in_grp"] > 0,
            batch_grp["breached_in_grp"] / batch_grp["assessable_in_grp"],
            0.0
        )
        batch_grp = batch_grp.rename(columns={"Batch_ID": "batch_id", "Batch_Date": "batch_date"})
        batch_grp["sla_group"] = group
        batch_grp["rolling_7d_avg_breach_rate"] = np.nan
        batch_grp["breach_rate_vs_trend_diff"] = np.nan
        batch_grp = batch_grp.sort_values("batch_date").reset_index(drop=True)

        result[group] = batch_grp[[
            "batch_id", "batch_date", "batch_breach_rate",
            "rolling_7d_avg_breach_rate", "breach_rate_vs_trend_diff", "sla_group"
        ]]

    return result


def build_pipeline_signals(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """Build batch-level pipeline observable signals."""
    gap_thresh: int   = cfg.get("pipeline_gap_days_threshold", 2)
    retry_thresh: int = cfg.get("pipeline_retry_threshold", 2)

    df_work = df.copy()
    df_work["Batch_Date"] = pd.to_datetime(df_work["Batch_Date"], errors="coerce")

    batch_pipeline = (
        df_work.groupby(["Batch_ID", "Batch_Date"])
        .agg(
            retry_count_sum       = ("Retry_Count",           "sum"),
            max_retry_count       = ("Retry_Count",           "max"),
            pipeline_gap          = ("Pipeline_Gap_Flag",     "any"),
            days_since_prev_batch = ("Days_Since_Prev_Batch", "first"),
        )
        .reset_index()
        .sort_values("Batch_Date")
        .reset_index(drop=True)
    )
    batch_pipeline = batch_pipeline.rename(columns={"Batch_ID": "batch_id", "Batch_Date": "batch_date"})

    statuses = []
    reasons = []
    for _, row in batch_pipeline.iterrows():
        if row["pipeline_gap"]:
            statuses.append("GAP_DETECTED")
            reasons.append(
                f"Pipeline_Gap_Flag detected for batch. "
                f"Days since previous batch: {row['days_since_prev_batch']}."
            )
        elif (
            pd.notna(row["days_since_prev_batch"])
            and row["days_since_prev_batch"] >= gap_thresh
        ) or (
            pd.notna(row["max_retry_count"])
            and row["max_retry_count"] >= retry_thresh
        ):
            parts = []
            if (
                pd.notna(row["days_since_prev_batch"])
                and row["days_since_prev_batch"] >= gap_thresh
            ):
                parts.append(
                    f"Days since previous batch ({row['days_since_prev_batch']}) "
                    f">= threshold ({gap_thresh})"
                )
            if (
                pd.notna(row["max_retry_count"])
                and row["max_retry_count"] >= retry_thresh
            ):
                parts.append(
                    f"Max retry count ({int(row['max_retry_count'])}) "
                    f">= threshold ({retry_thresh})"
                )
            statuses.append("DEGRADED")
            reasons.append("Pipeline degradation signal: " + "; ".join(parts) + ".")
        else:
            statuses.append("NORMAL")
            reasons.append("Pipeline operating within normal parameters.")

    batch_pipeline["pipeline_status"] = statuses
    batch_pipeline["reason"] = reasons

    return batch_pipeline


"""
sla_baseline.py
===============
Historical baseline computation for Temporal / SLA Monitoring.

Design requirements
-------------------
1. Chronological ordering — baselines are computed at each timepoint t using
   only observations from t-1 and earlier (no future leakage).
2. Segmented by population — separate baselines are computed for whole-batch
   metrics and group-specific breach rates.
3. Multi-Tier Robust Scale (Handling MAD = 0):
   - Tier 1: If MAD > 0, robust_sigma = MAD * 1.4826.
   - Tier 2: If MAD == 0, use sample standard deviation s of past observations in
     [t-window, t-1]. If s > 0, robust_sigma = s.
   - Tier 3: If s == 0 (entire historical window up to t-1 is completely constant),
     use the configured min_scale_floor (default 0.01) to prevent division-by-zero
     or spurious threshold triggering on numerical noise.
4. Minimum observations — when fewer than ``baseline_min_obs`` past points are
   available, baseline and sigma are returned as NaN (EWMA/CUSUM signals are
   suppressed downstream).
5. Reproducible — given the same input, output is strictly deterministic.
6. Pure computation — no I/O in this module.

Constants
---------
CONSISTENCY_FACTOR : float
    1.4826 — the factor that scales MAD to be a consistent estimator of sigma
    for a normal distribution.
"""


import logging
from typing import Optional

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

CONSISTENCY_FACTOR: float = 1.4826  # MAD -> robust sigma scaling constant


# ---------------------------------------------------------------------------
# Core rolling baseline utilities
# ---------------------------------------------------------------------------

def rolling_median_mad(
    values: np.ndarray,
    window: int,
    min_obs: int,
    min_scale_floor: float = 0.01,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute rolling historical median, MAD, and robust sigma with no future leakage.

    At each position t the window covers [t-window, t-1] (strictly prior).

    Parameters
    ----------
    values:
        1-D array of observations in chronological order.
    window:
        Maximum number of past observations to include.
    min_obs:
        Minimum past observations required; below this, NaN is returned.
    min_scale_floor:
        Configurable scale floor applied when historical window has zero variance.

    Returns
    -------
    (medians, mads, robust_sigmas) — three arrays of the same length as ``values``.
    """
    n = len(values)
    medians = np.full(n, np.nan)
    mads = np.full(n, np.nan)
    sigmas = np.full(n, np.nan)

    for t in range(n):
        start = max(0, t - window)
        past = values[start:t]  # strictly prior to t, no future leakage
        # Filter out NaNs if present
        valid_past = past[~np.isnan(past)]
        if len(valid_past) < min_obs:
            continue

        med = float(np.median(valid_past))
        mad = float(np.median(np.abs(valid_past - med)))

        # Multi-tier scale estimation
        if mad > 0:
            sigma = mad * CONSISTENCY_FACTOR
        else:
            # MAD == 0: check sample standard deviation
            if len(valid_past) > 1:
                std = float(np.std(valid_past, ddof=1))
            else:
                std = 0.0

            if std > 0:
                sigma = std
            else:
                # Completely constant historical series up to t-1
                sigma = float(min_scale_floor)

        medians[t] = med
        mads[t] = mad
        sigmas[t] = sigma

    return medians, mads, sigmas


def compute_series_baseline(
    ts: pd.DataFrame,
    value_col: str,
    window: int,
    min_obs: int,
    min_scale_floor: float = 0.01,
) -> pd.DataFrame:
    """Compute rolling historical median and robust sigma on a sorted time-series DataFrame.

    Parameters
    ----------
    ts:
        Chronologically sorted DataFrame.
    value_col:
        Column name containing metric values.
    window:
        Rolling window size.
    min_obs:
        Minimum past observations required.
    min_scale_floor:
        Floor for scale when variance is zero.

    Returns
    -------
    pd.DataFrame
        DataFrame with baseline_median, baseline_mad, baseline_robust_sigma appended.
    """
    out = ts.copy().sort_values("batch_date").reset_index(drop=True)
    if value_col not in out.columns or out.empty:
        out["baseline_median"] = np.nan
        out["baseline_mad"] = np.nan
        out["baseline_robust_sigma"] = np.nan
        return out

    values = out[value_col].to_numpy(dtype=float)
    medians, mads, sigmas = rolling_median_mad(
        values, window=window, min_obs=min_obs, min_scale_floor=min_scale_floor
    )

    out["baseline_median"] = medians
    out["baseline_mad"] = mads
    out["baseline_robust_sigma"] = sigmas

    return out


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def compute_timeliness_baseline(
    breach_series_dict: dict[str, pd.DataFrame],
    window: int,
    min_obs: int,
    min_scale_floor: float = 0.01,
) -> dict[str, pd.DataFrame]:
    """Compute rolling historical median/MAD baseline for whole-batch and group breach rates.

    Parameters
    ----------
    breach_series_dict:
        Dict mapping series key (e.g. 'WHOLE_BATCH', 'MEDICAL_CLAIM', etc.) to
        chronologically sorted DataFrames with column ``batch_breach_rate``.
    window:
        Rolling window size.
    min_obs:
        Minimum past observations required.
    min_scale_floor:
        Scale floor for constant historical series.

    Returns
    -------
    dict[str, pd.DataFrame]
        Updated DataFrames with baseline columns.
    """
    result: dict[str, pd.DataFrame] = {}
    for key, ts in breach_series_dict.items():
        result[key] = compute_series_baseline(
            ts=ts,
            value_col="batch_breach_rate",
            window=window,
            min_obs=min_obs,
            min_scale_floor=min_scale_floor,
        )
        logger.debug("Timeliness baseline computed for key='%s': %d rows", key, len(ts))
    return result


def compute_volume_baseline(
    volume_series: pd.DataFrame,
    window: int,
    min_obs: int,
    min_scale_floor: float = 0.01,
) -> pd.DataFrame:
    """Compute rolling historical median/MAD baseline for Volume_Vs_Trend_Ratio.

    Parameters
    ----------
    volume_series:
        Chronologically sorted DataFrame with column ``volume_ratio``
        (Volume_Vs_Trend_Ratio).
    window:
        Rolling window size.
    min_obs:
        Minimum past observations required.
    min_scale_floor:
        Scale floor for constant historical series.

    Returns
    -------
    pd.DataFrame
        Input with baseline_median, baseline_mad, baseline_robust_sigma added.
    """
    return compute_series_baseline(
        ts=volume_series,
        value_col="volume_ratio",
        window=window,
        min_obs=min_obs,
        min_scale_floor=min_scale_floor,
    )


"""
sla_ewma.py
===========
Exponentially Weighted Moving Average (EWMA) engine.

Purpose
-------
Detect *gradual* changes in SLA-related time-series behavior relative to a
historical baseline.

Formula
-------
    EWMA_t = alpha * x_t + (1 - alpha) * EWMA_{t-1}

Initialisation
--------------
    EWMA is seeded to the median of the first ``warmup_n`` observations to
    avoid an arbitrary starting point.  If fewer than ``warmup_n`` observations
    exist, the first available value is used.

Signals
-------
    NORMAL  : abs(EWMA_t - baseline_t) < warning_sigma * robust_sigma
    WARNING : warning_sigma * robust_sigma <= abs(...) < alert_sigma * robust_sigma
    ALERT   : abs(...) >= alert_sigma * robust_sigma

CRITICAL CONSTRAINT
-------------------
    An EWMA ALERT or WARNING is a *statistical* signal of process change.
    It is NEVER automatically classified as an SLA breach.

This module contains pure mathematical logic only (no I/O, no dataframe I/O).
"""


import logging
from dataclasses import dataclass, field
from typing import Sequence

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

# Signal constants
SIGNAL_NORMAL  = "NORMAL"
SIGNAL_WARNING = "WARNING"
SIGNAL_ALERT   = "ALERT"
SIGNAL_INSUFFICIENT_DATA = "INSUFFICIENT_DATA"


@dataclass
class EWMAResult:
    """Per-observation EWMA result."""
    raw_value:      float
    baseline:       float | None        # historical median at this timepoint
    ewma_value:     float | None        # EWMA statistic
    deviation:      float | None        # EWMA - baseline
    robust_sigma:   float | None        # baseline MAD × 1.4826
    ewma_signal:    str                 # NORMAL / WARNING / ALERT / INSUFFICIENT_DATA


def _classify_signal(
    deviation: float,
    robust_sigma: float,
    warning_sigma: float,
    alert_sigma: float,
) -> str:
    """Classify deviation relative to robust sigma thresholds.

    A one-sided test is used (positive deviation = deterioration for
    Batch_SLA_Breach_Rate; both sides used for Volume_Vs_Trend_Ratio).
    """
    if robust_sigma <= 0:
        # Constant baseline or insufficient variance — treat as no signal
        if abs(deviation) < 1e-9:
            return SIGNAL_NORMAL
        # Any nonzero deviation from a perfectly constant baseline is notable
        return SIGNAL_WARNING

    ratio = abs(deviation) / robust_sigma
    if ratio >= alert_sigma:
        return SIGNAL_ALERT
    if ratio >= warning_sigma:
        return SIGNAL_WARNING
    return SIGNAL_NORMAL


def compute_ewma_series(
    values: Sequence[float],
    baselines: Sequence[float | None],
    robust_sigmas: Sequence[float | None],
    alpha: float,
    warmup_n: int,
    warning_sigma: float,
    alert_sigma: float,
) -> list[EWMAResult]:
    """Compute EWMA for a chronologically ordered series.

    Parameters
    ----------
    values:
        Observed metric values in chronological order.
    baselines:
        Historical median at each timepoint (None = insufficient history).
    robust_sigmas:
        Historical MAD × 1.4826 at each timepoint (None = insufficient history).
    alpha:
        Smoothing factor (0 < alpha ≤ 1).
    warmup_n:
        Number of initial observations used to seed the EWMA.
    warning_sigma:
        Warning threshold in units of robust sigma.
    alert_sigma:
        Alert threshold in units of robust sigma.

    Returns
    -------
    list[EWMAResult]
        One result per observation.
    """
    if not 0 < alpha <= 1:
        raise ValueError(f"EWMA alpha must be in (0, 1], got {alpha}")

    n = len(values)
    results: list[EWMAResult] = []

    # Seed EWMA from warmup window (median of first warmup_n values)
    warmup = [v for v in values[:warmup_n] if not np.isnan(float(v))]
    ewma_prev: float | None = float(np.median(warmup)) if warmup else None

    for t, (x, base, sigma) in enumerate(zip(values, baselines, robust_sigmas)):
        x_f = float(x) if not np.isnan(float(x)) else np.nan

        # Update EWMA
        if ewma_prev is None:
            ewma_cur: float | None = x_f if not np.isnan(x_f) else None
        elif np.isnan(x_f):
            ewma_cur = ewma_prev  # carry forward on missing observation
        else:
            ewma_cur = alpha * x_f + (1.0 - alpha) * ewma_prev

        if ewma_cur is not None:
            ewma_prev = ewma_cur

        # Signal classification
        if base is None or np.isnan(float(base)):
            signal = SIGNAL_INSUFFICIENT_DATA
            deviation = None
        elif ewma_cur is None:
            signal = SIGNAL_INSUFFICIENT_DATA
            deviation = None
        else:
            deviation = ewma_cur - float(base)
            s = float(sigma) if sigma is not None and not np.isnan(float(sigma)) else 0.0
            signal = _classify_signal(deviation, s, warning_sigma, alert_sigma)

        results.append(
            EWMAResult(
                raw_value    = x_f,
                baseline     = float(base) if base is not None and not np.isnan(float(base)) else None,
                ewma_value   = ewma_cur,
                deviation    = deviation,
                robust_sigma = float(sigma) if sigma is not None and not np.isnan(float(sigma)) else None,
                ewma_signal  = signal,
            )
        )

    return results


def apply_ewma_to_dataframe(
    ts: pd.DataFrame,
    value_col: str,
    baseline_col: str,
    sigma_col: str,
    alpha: float,
    warmup_n: int,
    warning_sigma: float,
    alert_sigma: float,
) -> pd.DataFrame:
    """Apply EWMA to a batch time-series DataFrame and append result columns.

    The DataFrame must already be sorted chronologically.

    Parameters
    ----------
    ts:
        Chronologically sorted DataFrame.
    value_col:
        Name of the observed metric column.
    baseline_col:
        Name of the historical median column.
    sigma_col:
        Name of the robust sigma column.
    alpha, warmup_n, warning_sigma, alert_sigma:
        EWMA parameters.

    Returns
    -------
    pd.DataFrame
        Input with additional columns:
        ewma_value, ewma_deviation, ewma_signal.
    """
    out = ts.copy()

    def _to_float_or_nan(series: pd.Series) -> list[float]:
        return [float(v) if pd.notna(v) else float("nan") for v in series]

    values   = _to_float_or_nan(out[value_col])
    bases    = [
        float(v) if pd.notna(v) else None
        for v in out[baseline_col]
    ]
    sigmas   = [
        float(v) if pd.notna(v) else None
        for v in out[sigma_col]
    ]

    ewma_results = compute_ewma_series(
        values       = values,
        baselines    = bases,
        robust_sigmas= sigmas,
        alpha        = alpha,
        warmup_n     = warmup_n,
        warning_sigma= warning_sigma,
        alert_sigma  = alert_sigma,
    )

    out["ewma_value"]     = [r.ewma_value  for r in ewma_results]
    out["ewma_deviation"] = [r.deviation   for r in ewma_results]
    out["ewma_signal"]    = [r.ewma_signal for r in ewma_results]

    return out


"""
sla_cusum.py
============
Cumulative Sum (CUSUM) control chart engine.

Purpose
-------
Detect *sustained* shifts away from normal process behavior.  Unlike EWMA,
which reacts to individual deviations, CUSUM accumulates evidence of a shift
over multiple consecutive observations.

Formulae
--------
    S_pos_t = max(0,  S_pos_{t-1} + (x_t - baseline_t) - k)
    S_neg_t = max(0,  S_neg_{t-1} - (x_t - baseline_t) - k)

    where:
        k = reference value (half allowable shift; default 0.5 × robust_sigma)
        h = decision threshold (default 5.0 × robust_sigma)

Signal
------
    NORMAL         : S_pos_t < h  AND  S_neg_t < h
    SHIFT_DETECTED : S_pos_t >= h  OR  S_neg_t >= h

    On SHIFT_DETECTED, the cumulative sums are reset to 0 to allow detection
    of subsequent shifts (fast-initial response / FIR reset approach).

CRITICAL CONSTRAINT
-------------------
    A CUSUM SHIFT_DETECTED is a statistical signal indicating a *sustained*
    process shift over multiple observations.
    It is NEVER automatically classified as an SLA breach.

Distinction from EWMA
---------------------
    - EWMA detects *gradual* changes (single-observation sensitivity).
    - CUSUM detects *sustained* shifts (requires cumulative evidence).
    - Both may signal simultaneously; they provide complementary information.

This module contains pure mathematical logic only (no I/O, no dataframe I/O).
"""


import logging
from dataclasses import dataclass
from typing import Sequence

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

# Signal constants
SIGNAL_NORMAL         = "NORMAL"
SIGNAL_SHIFT_DETECTED = "SHIFT_DETECTED"
SIGNAL_INSUFFICIENT   = "INSUFFICIENT_DATA"


@dataclass
class CUSUMResult:
    """Per-observation CUSUM result."""
    raw_value:      float
    baseline:       float | None
    cusum_positive: float          # S_pos at this timepoint
    cusum_negative: float          # S_neg at this timepoint
    cusum_signal:   str            # NORMAL / SHIFT_DETECTED / INSUFFICIENT_DATA


def compute_cusum_series(
    values:       Sequence[float],
    baselines:    Sequence[float | None],
    robust_sigmas:Sequence[float | None],
    k_sigma:      float,
    h_sigma:      float,
) -> list[CUSUMResult]:
    """Compute two-sided CUSUM for a chronologically ordered series.

    Parameters
    ----------
    values:
        Observed metric values in chronological order.
    baselines:
        Historical median at each timepoint (None = insufficient history).
    robust_sigmas:
        Historical MAD × 1.4826 at each timepoint (None = insufficient history).
    k_sigma:
        Reference value as a multiple of robust sigma (e.g. 0.5).
    h_sigma:
        Decision threshold as a multiple of robust sigma (e.g. 5.0).

    Returns
    -------
    list[CUSUMResult]
        One result per observation.
    """
    n = len(values)
    results: list[CUSUMResult] = []
    s_pos: float = 0.0
    s_neg: float = 0.0

    for x, base, sigma in zip(values, baselines, robust_sigmas):
        x_f = float(x) if not np.isnan(float(x)) else np.nan

        if base is None or np.isnan(float(base)):
            results.append(
                CUSUMResult(
                    raw_value      = x_f,
                    baseline       = None,
                    cusum_positive = s_pos,
                    cusum_negative = s_neg,
                    cusum_signal   = SIGNAL_INSUFFICIENT,
                )
            )
            continue

        base_f  = float(base)
        sigma_f = float(sigma) if sigma is not None and not np.isnan(float(sigma)) else 0.0
        k       = k_sigma  * sigma_f
        h       = h_sigma  * sigma_f

        if np.isnan(x_f):
            # Missing observation: do not accumulate evidence; carry forward
            results.append(
                CUSUMResult(
                    raw_value      = x_f,
                    baseline       = base_f,
                    cusum_positive = s_pos,
                    cusum_negative = s_neg,
                    cusum_signal   = SIGNAL_NORMAL if (s_pos < h and s_neg < h) else SIGNAL_SHIFT_DETECTED,
                )
            )
            continue

        deviation = x_f - base_f

        # Accumulate
        s_pos = max(0.0,  s_pos + deviation - k)
        s_neg = max(0.0,  s_neg - deviation - k)

        # Classify
        if h > 0 and (s_pos >= h or s_neg >= h):
            signal = SIGNAL_SHIFT_DETECTED
            # FIR reset: zero cumulative sums after detection
            s_pos = 0.0
            s_neg = 0.0
        else:
            signal = SIGNAL_NORMAL

        results.append(
            CUSUMResult(
                raw_value      = x_f,
                baseline       = base_f,
                cusum_positive = s_pos,
                cusum_negative = s_neg,
                cusum_signal   = signal,
            )
        )

    return results


def apply_cusum_to_dataframe(
    ts:           pd.DataFrame,
    value_col:    str,
    baseline_col: str,
    sigma_col:    str,
    k_sigma:      float,
    h_sigma:      float,
) -> pd.DataFrame:
    """Apply CUSUM to a batch time-series DataFrame and append result columns.

    The DataFrame must already be sorted chronologically.

    Parameters
    ----------
    ts:
        Chronologically sorted DataFrame.
    value_col:
        Name of the observed metric column.
    baseline_col:
        Name of the historical median column.
    sigma_col:
        Name of the robust sigma column.
    k_sigma, h_sigma:
        CUSUM reference and threshold multipliers.

    Returns
    -------
    pd.DataFrame
        Input with additional columns:
        cusum_positive, cusum_negative, cusum_signal.
    """
    out = ts.copy()

    def _to_float_or_nan(series: pd.Series) -> list[float]:
        return [float(v) if pd.notna(v) else float("nan") for v in series]

    values  = _to_float_or_nan(out[value_col])
    bases   = [float(v) if pd.notna(v) else None for v in out[baseline_col]]
    sigmas  = [float(v) if pd.notna(v) else None for v in out[sigma_col]]

    cusum_results = compute_cusum_series(
        values        = values,
        baselines     = bases,
        robust_sigmas = sigmas,
        k_sigma       = k_sigma,
        h_sigma       = h_sigma,
    )

    out["cusum_positive"] = [r.cusum_positive for r in cusum_results]
    out["cusum_negative"] = [r.cusum_negative for r in cusum_results]
    out["cusum_signal"]   = [r.cusum_signal   for r in cusum_results]

    return out


"""
sla_breach_detection.py
=======================
Deterministic SLA breach classification at the record level.

This module implements the strict separation between:

    1. DATA QUALITY
       "Is the data valid enough to interpret?"
       -> Handled by temporal_validity classification in sla_metrics.py
         and by existing R016 in quality_engine.py.

    2. SLA BREACH
       "Has the applicable SLA target actually been exceeded?"
       -> Handled here. Deterministic; requires valid temporal data.

    3. SLA RISK
       "Is the process showing evidence of deterioration that could
       threaten the applicable SLA?"
       -> Handled here, combining EWMA + CUSUM signals with timeliness.

    4. TEMPORAL ANOMALY (EWMA / CUSUM)
       -> Statistical signals from sla_ewma.py and sla_cusum.py.
       -> NEVER automatically classified as SLA breach here.
"""


import logging
from typing import Any

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

# Sentinel values
NOT_ASSESSABLE = "NOT_ASSESSABLE"

# Status labels
STATUS_NORMAL         = "NORMAL"
STATUS_AT_RISK        = "AT_RISK"
STATUS_BREACHED       = "BREACHED"
STATUS_NOT_ASSESSABLE = "NOT_ASSESSABLE"

# SLA risk levels
RISK_LOW    = "LOW"
RISK_MEDIUM = "MEDIUM"
RISK_HIGH   = "HIGH"

# EWMA / CUSUM signal constants
EWMA_NORMAL  = "NORMAL"
EWMA_WARNING = "WARNING"
EWMA_ALERT   = "ALERT"
CUSUM_NORMAL = "NORMAL"
CUSUM_SHIFT  = "SHIFT_DETECTED"


# ---------------------------------------------------------------------------
# Record-level breach determination
# ---------------------------------------------------------------------------

def determine_record_breach(
    temporal_validity: str,
    processing_latency_days: float | None,
    sla_target_days: float | None,
) -> dict[str, Any]:
    """Determine SLA breach status for a single record."""
    # --- Data quality problems -> NOT_ASSESSABLE ----------------------------
    if temporal_validity == "NEGATIVE":
        return {
            "sla_breach":      NOT_ASSESSABLE,
            "sla_utilization": None,
            "status":          STATUS_NOT_ASSESSABLE,
            "reason": (
                f"Negative processing latency "
                f"({processing_latency_days} days) indicates an invalid "
                f"temporal relationship (Processed_Date earlier than "
                f"Service_Date). SLA assessment cannot be performed. "
                f"See R016."
            ),
        }

    if temporal_validity in ("NULL_NO_DATE", "MISSING_LATENCY"):
        return {
            "sla_breach":      NOT_ASSESSABLE,
            "sla_utilization": None,
            "status":          STATUS_NOT_ASSESSABLE,
            "reason": (
                "No Processed_Date or Processing_Latency_Days available. "
                "SLA assessment cannot be performed."
            ),
        }

    # --- Valid temporal data -----------------------------------------------
    if sla_target_days is None or np.isnan(float(sla_target_days)):
        return {
            "sla_breach":      NOT_ASSESSABLE,
            "sla_utilization": None,
            "status":          STATUS_NOT_ASSESSABLE,
            "reason": "SLA_Target_Days is missing; cannot determine applicable SLA.",
        }

    latency  = float(processing_latency_days)
    target   = float(sla_target_days)
    util     = round(latency / target, 4) if target > 0 else None

    if latency > target:
        return {
            "sla_breach":      True,
            "sla_utilization": util,
            "status":          STATUS_BREACHED,
            "reason": (
                f"Processing latency ({latency} days) exceeds SLA target "
                f"({target} days). SLA breach confirmed."
            ),
        }

    return {
        "sla_breach":      False,
        "sla_utilization": util,
        "status":          STATUS_NORMAL,  # may be upgraded to AT_RISK by risk module
        "reason": (
            f"Processing latency ({latency} days) within SLA target "
            f"({target} days). Utilisation: {util:.1%}."
        ),
    }


# ---------------------------------------------------------------------------
# Record-level SLA risk classification
# ---------------------------------------------------------------------------

def classify_record_sla_risk(
    breach_result: dict[str, Any],
    ewma_signal: str,
    cusum_signal: str,
) -> dict[str, Any]:
    """Upgrade status to AT_RISK when statistical signals indicate deterioration."""
    result = dict(breach_result)
    ewma  = ewma_signal  or EWMA_NORMAL
    cusum = cusum_signal or CUSUM_NORMAL

    # NOT_ASSESSABLE records: no risk classification
    if result["sla_breach"] == NOT_ASSESSABLE:
        result["sla_risk"]     = None
        result["risk_signals"] = {"ewma_signal": ewma, "cusum_signal": cusum}
        return result

    # BREACHED records: already worst-case
    if result["sla_breach"] is True:
        result["sla_risk"]     = None
        result["risk_signals"] = {"ewma_signal": ewma, "cusum_signal": cusum}
        return result

    # --- VALID, not breached -> assess risk from statistical signals ---------
    both_shift    = (ewma == EWMA_ALERT and cusum == CUSUM_SHIFT)
    partial_shift = (ewma in (EWMA_WARNING, EWMA_ALERT) or cusum == CUSUM_SHIFT)

    if both_shift:
        risk   = RISK_HIGH
        status = STATUS_AT_RISK
        reason = (
            result["reason"] + " However, EWMA signals ALERT and CUSUM signals "
            "SHIFT_DETECTED, indicating sustained process deterioration. "
            "SLA breach risk is HIGH."
        )
    elif partial_shift:
        risk   = RISK_MEDIUM if ewma != EWMA_ALERT else RISK_HIGH
        status = STATUS_AT_RISK
        reason = (
            result["reason"] + " However, statistical monitoring signals "
            f"(EWMA={ewma}, CUSUM={cusum}) indicate process deterioration. "
            f"SLA breach risk is {risk}."
        )
    else:
        risk   = RISK_LOW
        status = STATUS_NORMAL
        reason = result["reason"]

    result["sla_risk"]     = risk
    result["status"]       = status
    result["reason"]       = reason
    result["risk_signals"] = {"ewma_signal": ewma, "cusum_signal": cusum}
    return result


# ---------------------------------------------------------------------------
# Batch-level SLA monitoring status
# ---------------------------------------------------------------------------

def classify_batch_sla_status(
    batch_breach_rate: float,
    rolling_7d_avg_breach_rate: float,
    ewma_signal: str,
    cusum_signal: str,
    pipeline_status: str,
) -> dict[str, Any]:
    """Classify batch-level SLA monitoring status."""
    ewma  = ewma_signal   or EWMA_NORMAL
    cusum = cusum_signal  or CUSUM_NORMAL
    pipe  = pipeline_status or STATUS_NORMAL

    signals: list[str] = []
    is_at_risk = False

    if ewma in (EWMA_WARNING, EWMA_ALERT):
        signals.append(f"EWMA={ewma}")
        is_at_risk = True

    if cusum == CUSUM_SHIFT:
        signals.append("CUSUM=SHIFT_DETECTED")
        is_at_risk = True

    if pipe in ("DEGRADED", "GAP_DETECTED"):
        signals.append(f"pipeline_status={pipe}")
        is_at_risk = True

    if is_at_risk:
        both_statistical = (ewma == EWMA_ALERT and cusum == CUSUM_SHIFT)
        if both_statistical:
            label = "Elevated Batch SLA Breach Rate — Sustained Deterioration"
        elif ewma in (EWMA_WARNING, EWMA_ALERT) or cusum == CUSUM_SHIFT:
            label = "Elevated Batch SLA Breach Rate — Early Deterioration Signal"
        else:
            label = "Pipeline Degradation Signal"

        reason = (
            f"Batch SLA breach rate: {batch_breach_rate:.2%} "
            f"(7-day avg: {rolling_7d_avg_breach_rate:.2%}). "
            f"Monitoring signals: {', '.join(signals)}. "
            f"This is a monitoring signal indicating potential process "
            f"deterioration, not a batch-level SLA breach determination "
            f"(no universal batch-level SLA threshold is defined)."
        )
        return {
            "batch_sla_status": STATUS_AT_RISK,
            "label":            label,
            "sla_breach":       False,
            "reason":           reason,
        }

    return {
        "batch_sla_status": STATUS_NORMAL,
        "label":            "Normal Batch SLA Performance",
        "sla_breach":       False,
        "reason": (
            f"Batch SLA breach rate: {batch_breach_rate:.2%} "
            f"(7-day avg: {rolling_7d_avg_breach_rate:.2%}). "
            "No abnormal monitoring signals detected."
        ),
    }


"""
sla_monitor.py
==============
Orchestrator for the Temporal / SLA Monitoring module.

Pipeline
--------
    Feature DataFrame
        |
        v
    SLA Metrics (sla_metrics.py)
        |
        +-- Volume Monitoring series (Volume_Vs_Trend_Ratio)
        |
        +-- Timeliness SLA series (whole-batch & group-specific)
        |
        +-- Pipeline Monitoring signals
        |
        v
    Historical Baseline (sla_baseline.py)
        |
        v
    EWMA (sla_ewma.py) & CUSUM (sla_cusum.py)
        |
        v
    Deterministic SLA Breach Detection & Risk Classification (sla_breach_detection.py)
        |
        v
    Temporal/SLA Findings -> outputs/sla_temporal_findings.json

Integration
-----------
Called from main.py *after* the existing quality engine and scoring pipeline.
Does not modify or replace any existing functionality.
"""


import datetime
import json
import logging
import os
from typing import Any

import numpy as np
import pandas as pd

# Internal module imports are inlined above in this single-file Colab version.

logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# JSON serialisation helpers
# ---------------------------------------------------------------------------

class _SafeEncoder(json.JSONEncoder):
    """JSON encoder that handles numpy types, NaN/Inf, and timestamps."""

    def default(self, obj: Any) -> Any:
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            if np.isnan(obj) or np.isinf(obj):
                return None
            return float(obj)
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        if isinstance(obj, pd.Timestamp):
            return obj.isoformat()
        if isinstance(obj, datetime.date):
            return obj.isoformat()
        return super().default(obj)

    def encode(self, obj: Any) -> str:  # type: ignore[override]
        def _clean(o: Any) -> Any:
            if isinstance(o, float) and (np.isnan(o) or np.isinf(o)):
                return None
            if isinstance(o, dict):
                return {k: _clean(v) for k, v in o.items()}
            if isinstance(o, list):
                return [_clean(v) for v in o]
            return o
        return super().encode(_clean(obj))


def _safe_val(v: Any) -> Any:
    """Convert numpy scalars and NaN to Python-native types."""
    if v is None:
        return None
    if isinstance(v, float) and (np.isnan(v) or np.isinf(v)):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return None if np.isnan(v) else float(v)
    return v


# ---------------------------------------------------------------------------
# Signal Lookup
# ---------------------------------------------------------------------------

def _build_signal_lookup(
    batch_stats_by_group: dict[str, pd.DataFrame]
) -> dict[tuple[str, str], dict[str, str]]:
    """Build a lookup from (sla_group, batch_id) -> {ewma_signal, cusum_signal}."""
    lookup: dict[tuple[str, str], dict[str, str]] = {}
    for group, ts in batch_stats_by_group.items():
        for _, row in ts.iterrows():
            key = (group, str(row["batch_id"]))
            lookup[key] = {
                "ewma_signal":  str(row.get("ewma_signal",  EWMA_NORMAL)),
                "cusum_signal": str(row.get("cusum_signal", CUSUM_NORMAL)),
            }
    return lookup


# ---------------------------------------------------------------------------
# Record-level findings
# ---------------------------------------------------------------------------

def _build_record_findings(
    enriched_df: pd.DataFrame,
    signal_lookup: dict[tuple[str, str], dict[str, str]],
) -> list[dict[str, Any]]:
    """Build per-record timeliness findings."""
    findings: list[dict[str, Any]] = []

    for _, row in enriched_df.iterrows():
        sla_group  = str(row.get("sla_group", "UNKNOWN"))
        batch_id   = str(row.get("Batch_ID", ""))
        validity   = str(row.get("temporal_validity", ""))
        latency    = row.get("Processing_Latency_Days")
        target     = row.get("SLA_Target_Days")

        breach_result = determine_record_breach(
            temporal_validity        = validity,
            processing_latency_days  = latency,
            sla_target_days          = target,
        )

        # Lookup group signals first, fallback to whole-batch signals
        signals = signal_lookup.get(
            (sla_group, batch_id),
            signal_lookup.get(("WHOLE_BATCH", batch_id), {})
        )
        ewma_sig  = signals.get("ewma_signal",  EWMA_NORMAL)
        cusum_sig = signals.get("cusum_signal", CUSUM_NORMAL)

        risk_result = classify_record_sla_risk(breach_result, ewma_sig, cusum_sig)

        finding: dict[str, Any] = {
            "record_id":               str(row.get("Record_ID", "")),
            "record_type":             str(row.get("Record_Type", "")),
            "sla_group":               sla_group,
            "batch_id":                batch_id,
            "batch_date":              str(row.get("Batch_Date", "")),
            "sla_target_days":         _safe_val(target),
            "processing_latency_days": _safe_val(latency),
            "temporal_validity":       validity,
            "sla_utilization":         _safe_val(risk_result.get("sla_utilization")),
            "sla_breach":              risk_result.get("sla_breach"),
            "sla_risk":                risk_result.get("sla_risk"),
            "status":                  risk_result.get("status"),
            "ewma_signal":             ewma_sig,
            "cusum_signal":            cusum_sig,
            "reason":                  risk_result.get("reason", ""),
        }
        findings.append(finding)

    return findings


# ---------------------------------------------------------------------------
# Batch-level findings
# ---------------------------------------------------------------------------

def _build_batch_findings(
    batch_stats_by_group: dict[str, pd.DataFrame],
    pipeline_df: pd.DataFrame,
) -> list[dict[str, Any]]:
    """Build per-batch monitoring findings."""
    pipeline_lookup = {
        str(row["batch_id"]): str(row["pipeline_status"])
        for _, row in pipeline_df.iterrows()
    }

    findings: list[dict[str, Any]] = []

    for group, ts in batch_stats_by_group.items():
        for _, row in ts.iterrows():
            batch_id    = str(row["batch_id"])
            pipe_status = pipeline_lookup.get(batch_id, STATUS_NORMAL)

            breach_rate = _safe_val(row.get("batch_breach_rate",     0.0)) or 0.0
            rolling_avg = _safe_val(row.get("rolling_7d_avg_breach_rate", 0.0)) or 0.0
            ewma_sig    = str(row.get("ewma_signal",  EWMA_NORMAL))
            cusum_sig   = str(row.get("cusum_signal", CUSUM_NORMAL))

            batch_status = classify_batch_sla_status(
                batch_breach_rate          = breach_rate,
                rolling_7d_avg_breach_rate = rolling_avg,
                ewma_signal                = ewma_sig,
                cusum_signal               = cusum_sig,
                pipeline_status            = pipe_status,
            )

            finding: dict[str, Any] = {
                "batch_id":                   batch_id,
                "batch_date":                 str(row.get("batch_date", "")),
                "sla_group":                  group,
                "metric":                     "batch_sla_breach_rate",
                "metric_value":               breach_rate,
                "rolling_7d_avg_breach_rate": rolling_avg,
                "breach_rate_vs_trend_diff":  _safe_val(row.get("breach_rate_vs_trend_diff")),
                "baseline_median":            _safe_val(row.get("baseline_median")),
                "baseline_mad":               _safe_val(row.get("baseline_mad")),
                "baseline_robust_sigma":      _safe_val(row.get("baseline_robust_sigma")),
                "ewma_value":                 _safe_val(row.get("ewma_value")),
                "ewma_deviation":             _safe_val(row.get("ewma_deviation")),
                "ewma_signal":                ewma_sig,
                "cusum_positive":             _safe_val(row.get("cusum_positive")),
                "cusum_negative":             _safe_val(row.get("cusum_negative")),
                "cusum_signal":               cusum_sig,
                "pipeline_status":            pipe_status,
                "batch_sla_status":           batch_status["batch_sla_status"],
                "label":                      batch_status["label"],
                "sla_breach":                 batch_status["sla_breach"],
                "reason":                     batch_status["reason"],
            }
            findings.append(finding)

    return findings


# ---------------------------------------------------------------------------
# Pipeline findings
# ---------------------------------------------------------------------------

def _build_pipeline_findings(pipeline_df: pd.DataFrame) -> list[dict[str, Any]]:
    findings: list[dict[str, Any]] = []
    for _, row in pipeline_df.iterrows():
        findings.append({
            "batch_id":              str(row["batch_id"]),
            "batch_date":            str(row["batch_date"]),
            "retry_count_sum":       _safe_val(row.get("retry_count_sum")),
            "max_retry_count":       _safe_val(row.get("max_retry_count")),
            "pipeline_gap":          bool(row.get("pipeline_gap", False)),
            "days_since_prev_batch": _safe_val(row.get("days_since_prev_batch")),
            "pipeline_status":       str(row["pipeline_status"]),
            "reason":                str(row["reason"]),
        })
    return findings


# ---------------------------------------------------------------------------
# Volume findings
# ---------------------------------------------------------------------------

def _build_volume_findings(vol_ts: pd.DataFrame) -> list[dict[str, Any]]:
    findings: list[dict[str, Any]] = []
    for _, row in vol_ts.iterrows():
        ratio    = _safe_val(row.get("volume_ratio"))
        baseline = _safe_val(row.get("baseline_median"))
        ewma_sig = str(row.get("ewma_signal", EWMA_NORMAL))

        if ewma_sig in ("WARNING", "ALERT"):
            vol_status = "VOLUME_DEVIATION_DETECTED"
            label      = f"Volume Deviation Detected (EWMA={ewma_sig})"
        else:
            vol_status = STATUS_NORMAL
            label      = "Normal Batch Volume"

        findings.append({
            "batch_id":              str(row["batch_id"]),
            "batch_date":            str(row["batch_date"]),
            "metric":                "Volume_Vs_Trend_Ratio",
            "actual_volume":         _safe_val(row.get("actual_volume")),
            "baseline_volume_7d":    _safe_val(row.get("baseline_volume")),
            "volume_ratio":          ratio,
            "volume_deviation":      _safe_val(row.get("volume_deviation")),
            "baseline_median":       baseline,
            "baseline_robust_sigma": _safe_val(row.get("baseline_robust_sigma")),
            "ewma_value":            _safe_val(row.get("ewma_value")),
            "ewma_deviation":        _safe_val(row.get("ewma_deviation")),
            "ewma_signal":           ewma_sig,
            "volume_status":         vol_status,
            "label":                 label,
            "sla_breach":            False,  # volume anomaly != SLA breach
        })
    return findings


# ---------------------------------------------------------------------------
# Dynamic Summary Statistics & Reconciliation
# ---------------------------------------------------------------------------

def _build_summary(
    record_findings: list[dict[str, Any]],
    batch_findings:  list[dict[str, Any]],
    pipeline_findings: list[dict[str, Any]],
    total_df_records: int,
) -> dict[str, Any]:
    """Calculate strictly reconciling summary metrics from findings."""
    total_records          = total_df_records
    records_not_assessable = sum(1 for f in record_findings if f["sla_breach"] == NOT_ASSESSABLE)
    records_assessable     = total_records - records_not_assessable
    records_breached       = sum(1 for f in record_findings if f["sla_breach"] is True)
    records_normal         = sum(1 for f in record_findings if f["sla_breach"] is False)
    records_at_risk        = sum(1 for f in record_findings if f.get("status") == STATUS_AT_RISK)

    # Reconciliation asserts (logs warning if ever violated)
    if records_breached + records_normal != records_assessable:
        logger.error(
            "Summary mismatch: breached (%d) + normal (%d) != assessable (%d)",
            records_breached, records_normal, records_assessable,
        )
    if records_assessable + records_not_assessable != total_records:
        logger.error(
            "Summary mismatch: assessable (%d) + not_assessable (%d) != total (%d)",
            records_assessable, records_not_assessable, total_records,
        )

    batches_at_risk = sum(
        1 for f in batch_findings if f.get("batch_sla_status") == STATUS_AT_RISK
    )
    pipeline_gaps = sum(
        1 for f in pipeline_findings if f.get("pipeline_status") == "GAP_DETECTED"
    )
    pipeline_degraded = sum(
        1 for f in pipeline_findings if f.get("pipeline_status") == "DEGRADED"
    )

    by_group: dict[str, dict[str, int]] = {}
    for f in record_findings:
        grp = f["sla_group"]
        if grp not in by_group:
            by_group[grp] = {
                "total": 0,
                "assessable": 0,
                "not_assessable": 0,
                "breached": 0,
                "normal": 0,
                "at_risk": 0,
            }
        by_group[grp]["total"] += 1
        if f["sla_breach"] == NOT_ASSESSABLE:
            by_group[grp]["not_assessable"] += 1
        else:
            by_group[grp]["assessable"] += 1
            if f["sla_breach"] is True:
                by_group[grp]["breached"] += 1
            else:
                by_group[grp]["normal"] += 1
            if f.get("status") == STATUS_AT_RISK:
                by_group[grp]["at_risk"] += 1

    return {
        "total_records":              total_records,
        "records_assessable":         records_assessable,
        "records_not_assessable":     records_not_assessable,
        "records_breached":           records_breached,
        "records_normal":             records_normal,
        "records_at_risk":            records_at_risk,
        "batches_total":              len(set(f["batch_id"] for f in batch_findings)),
        "batches_at_risk":            batches_at_risk,
        "pipeline_gaps_detected":     pipeline_gaps,
        "pipeline_degraded_batches":  pipeline_degraded,
        "by_sla_group":               by_group,
    }


# ---------------------------------------------------------------------------
# Main Entry Point
# ---------------------------------------------------------------------------

def run_sla_monitoring(
    df: pd.DataFrame,
    config_overrides: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Run the full Temporal / SLA Monitoring pipeline."""
    cfg = get_config(config_overrides)
    logger.info("Starting Temporal / SLA Monitoring pipeline.")

    # 1. Work on a copy
    df_work = df.copy()
    if "Batch_Date" in df_work.columns:
        df_work["Batch_Date"] = pd.to_datetime(df_work["Batch_Date"], errors="coerce")

    # 2. Extract metrics
    enriched_df            = extract_record_timeliness_metrics(df_work)
    vol_series             = build_batch_volume_series(df_work)
    breach_series_by_group = build_batch_breach_rate_series(df_work)
    pipeline_df            = build_pipeline_signals(df_work, cfg)

    # 3. Compute baselines
    window          = cfg["baseline_window"]
    min_obs         = cfg["baseline_min_obs"]
    min_scale_floor = cfg.get("min_scale_floor", 0.01)

    baseline_by_group = compute_timeliness_baseline(
        breach_series_by_group, window=window, min_obs=min_obs, min_scale_floor=min_scale_floor
    )

    if not vol_series.empty:
        vol_series = compute_volume_baseline(
            vol_series, window=window, min_obs=min_obs, min_scale_floor=min_scale_floor
        )

    # 4. EWMA
    alpha       = cfg["ewma_alpha"]
    warmup_n    = cfg["ewma_warmup_n"]
    warn_sigma  = cfg["ewma_warning_sigma"]
    alert_sigma = cfg["ewma_alert_sigma"]

    ewma_by_group: dict[str, pd.DataFrame] = {}
    for group, ts in baseline_by_group.items():
        ewma_by_group[group] = apply_ewma_to_dataframe(
            ts            = ts,
            value_col     = "batch_breach_rate",
            baseline_col  = "baseline_median",
            sigma_col     = "baseline_robust_sigma",
            alpha         = alpha,
            warmup_n      = warmup_n,
            warning_sigma = warn_sigma,
            alert_sigma   = alert_sigma,
        )

    if not vol_series.empty and "baseline_median" in vol_series.columns:
        vol_series = apply_ewma_to_dataframe(
            ts            = vol_series,
            value_col     = "volume_ratio",
            baseline_col  = "baseline_median",
            sigma_col     = "baseline_robust_sigma",
            alpha         = alpha,
            warmup_n      = warmup_n,
            warning_sigma = warn_sigma,
            alert_sigma   = alert_sigma,
        )

    # 5. CUSUM
    k_sigma = cfg["cusum_k_sigma"]
    h_sigma = cfg["cusum_h_sigma"]

    cusum_by_group: dict[str, pd.DataFrame] = {}
    for group, ts in ewma_by_group.items():
        cusum_by_group[group] = apply_cusum_to_dataframe(
            ts            = ts,
            value_col     = "batch_breach_rate",
            baseline_col  = "baseline_median",
            sigma_col     = "baseline_robust_sigma",
            k_sigma       = k_sigma,
            h_sigma       = h_sigma,
        )

    # 6. Signals lookup & findings generation
    signal_lookup     = _build_signal_lookup(cusum_by_group)
    record_findings   = _build_record_findings(enriched_df, signal_lookup)
    batch_findings    = _build_batch_findings(cusum_by_group, pipeline_df)
    pipeline_findings = _build_pipeline_findings(pipeline_df)
    volume_findings   = _build_volume_findings(vol_series) if not vol_series.empty else []
    summary           = _build_summary(record_findings, batch_findings, pipeline_findings, len(df))

    # 7. Output structure
    output: dict[str, Any] = {
        "run_timestamp":       datetime.datetime.now().isoformat(),
        "config": {
            "baseline_window":             cfg["baseline_window"],
            "baseline_min_obs":            cfg["baseline_min_obs"],
            "min_scale_floor":             cfg.get("min_scale_floor", 0.01),
            "ewma_alpha":                  cfg["ewma_alpha"],
            "ewma_warning_sigma":          cfg["ewma_warning_sigma"],
            "ewma_alert_sigma":            cfg["ewma_alert_sigma"],
            "cusum_k_sigma":               cfg["cusum_k_sigma"],
            "cusum_h_sigma":               cfg["cusum_h_sigma"],
            "pipeline_gap_days_threshold": cfg["pipeline_gap_days_threshold"],
            "pipeline_retry_threshold":    cfg["pipeline_retry_threshold"],
        },
        "sla_group_targets": {
            "MEDICAL_CLAIM":        30,
            "PHARMACY_CLAIM":       2,
            "PRIOR_AUTH_STANDARD":  14,
            "PRIOR_AUTH_EXPEDITED": 3,
        },
        "summary":                summary,
        "record_level_findings":  record_findings,
        "batch_level_findings":   batch_findings,
        "pipeline_findings":      pipeline_findings,
        "volume_findings":        volume_findings,
    }

    # Write output
    output_dir = cfg["output_dir"]
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, cfg["findings_filename"])
    with open(out_path, "w") as fh:
        json.dump(output, fh, indent=2, cls=_SafeEncoder)

    print(f"Temporal/SLA findings saved to {out_path}")
    print(
        f"  Total records:          {summary['total_records']}\n"
        f"  Records assessable:     {summary['records_assessable']}\n"
        f"  Records not-assessable: {summary['records_not_assessable']}\n"
        f"  Records breached:       {summary['records_breached']}\n"
        f"  Records normal:         {summary['records_normal']}\n"
        f"  Records at-risk:        {summary['records_at_risk']}\n"
        f"  Batches at-risk:        {summary['batches_at_risk']}\n"
        f"  Pipeline gaps:          {summary['pipeline_gaps_detected']}"
    )

    return output


# ============================================================
# GOOGLE COLAB ENTRY POINT
# ============================================================
# This SLA pipeline is downstream of:
#   1. Feature Engineering
#   2. Data Quality Engine
#
# It reads:
#   claims_pharmacy_auth_monitor_dataset_features.csv
#   outputs/quality_report.json
#
# It writes:
#   outputs/sla_temporal_findings.json
#
# The quality_report.json is read only as the upstream
# Data Quality Engine output/context. It does not change
# the existing SLA calculations.
# ============================================================

DATA_PATH = "claims_pharmacy_auth_monitor_dataset_features.csv"
QUALITY_REPORT_PATH = "outputs/quality_report.json"

print("=" * 70)
print("UC10 - SLA / TEMPORAL MONITORING")
print("=" * 70)

# ------------------------------------------------------------
# 1. Confirm Data Quality Engine completed
# ------------------------------------------------------------

if not os.path.exists(QUALITY_REPORT_PATH):
    raise FileNotFoundError(
        "outputs/quality_report.json was not found. "
        "Run the Data Quality Engine first."
    )

with open(QUALITY_REPORT_PATH, "r") as f:
    quality_report = json.load(f)

print()
print("UPSTREAM DATA QUALITY RESULT")
print("-" * 70)
print(f"Records scanned       : {quality_report.get('records_scanned')}")
print(
    f"Overall quality score : "
    f"{quality_report.get('overall_quality_score'):.2f}"
)
print(f"Overall risk level    : {quality_report.get('overall_risk_level')}")
print(f"Critical issues       : {quality_report.get('critical_issue_count')}")
print()
print("Data Quality Engine output found.")
print("Starting separate SLA pipeline...")

# ------------------------------------------------------------
# 2. Load feature-engineered dataset
# ------------------------------------------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. "
        "Run the Feature Engineering pipeline first."
    )

dtype_spec = {
    "Record_ID": str,
    "BENE_ID": str,
    "Provider_NPI": str,
    "Auth_Linked_ID": str,
}

print()
print(f"Loading feature dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, dtype=dtype_spec)

date_columns = [
    "Service_Date",
    "Service_End_Date",
    "Processed_Date",
    "Decision_Date",
    "Submission_Date",
    "Batch_Date",
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"Loaded {len(df):,} rows, {len(df.columns)} columns.")

# ------------------------------------------------------------
# 3. Check that the feature-engineering output contains
#    the columns required by the existing SLA code.
# ------------------------------------------------------------

required_sla_columns = [
    "Record_ID",
    "Record_Type",
    "Batch_ID",
    "Batch_Date",
    "Processed_Date",
    "Processing_Latency_Days",
    "SLA_Target_Days",
    "Urgency_Flag",
    "Batch_SLA_Breach_Rate",
    "Rolling_7D_Avg_SLA_Breach_Rate",
    "SLA_Breach_Rate_Vs_Trend_Diff",
    "Batch_Volume",
    "Rolling_7D_Avg_Volume",
    "Volume_Vs_Trend_Ratio",
    "Retry_Count",
    "Pipeline_Gap_Flag",
    "Days_Since_Prev_Batch",
]

missing_sla_columns = [
    c for c in required_sla_columns
    if c not in df.columns
]

if missing_sla_columns:
    raise ValueError(
        "The feature-engineered dataset is missing required SLA columns:\n"
        + "\n".join(f" - {c}" for c in missing_sla_columns)
    )

# ------------------------------------------------------------
# 4. Run ONLY the SLA / Temporal Monitoring pipeline
# ------------------------------------------------------------

sla_output = run_sla_monitoring(df)

# ------------------------------------------------------------
# 5. Final confirmation
# ------------------------------------------------------------

print()
print("=" * 70)
print("SLA PIPELINE COMPLETED")
print("=" * 70)
print()
print("Output:")
print("outputs/sla_temporal_findings.json")
print()


# Optional Colab download:
#
# from google.colab import files
# files.download("outputs/sla_temporal_findings.json")

UC10 - SLA / TEMPORAL MONITORING

UPSTREAM DATA QUALITY RESULT
----------------------------------------------------------------------
Records scanned       : 10000
Overall quality score : 91.34
Overall risk level    : MEDIUM
Critical issues       : 2

Data Quality Engine output found.
Starting separate SLA pipeline...

Loading feature dataset: claims_pharmacy_auth_monitor_dataset_features.csv
Loaded 10,000 rows, 50 columns.
Temporal/SLA findings saved to outputs/sla_temporal_findings.json
  Total records:          10000
  Records assessable:     6969
  Records not-assessable: 3031
  Records breached:       1587
  Records normal:         5382
  Records at-risk:        75
  Batches at-risk:        3891
  Pipeline gaps:          21

SLA PIPELINE COMPLETED

Output:
outputs/sla_temporal_findings.json



In [ ]:
# ============================================================
# MODULE 1: Statistical Detection (Z-score + IQR)
# Runs after the Feature Engineering cell - uses `df` already
# in memory. No files to upload.
#
# Applies Z-score and IQR outlier detection to key numeric
# fields, computed WITHIN each Record_Type group (medical claims,
# pharmacy claims, and prior auths have very different scales -
# mixing them would make legitimate pharmacy amounts look like
# outliers next to medical claim amounts, and vice versa).
# ============================================================

import pandas as pd
import numpy as np
import json
import os

OUTPUT_DIR = "outputs"
OUTPUT_PATH = f"{OUTPUT_DIR}/statistical_findings.json"

# Fields checked per record type (only fields that apply to that type)
FIELDS_BY_TYPE = {
    "MEDICAL_CLAIM":   ["Billed_Amount", "Allowed_Amount", "Paid_Amount",
                         "Patient_Responsibility", "Processing_Latency_Days"],
    "PHARMACY_CLAIM":  ["Billed_Amount", "Allowed_Amount", "Paid_Amount",
                         "Patient_Responsibility", "Days_Supply",
                         "Quantity_Dispensed", "Processing_Latency_Days"],
    "PRIOR_AUTH":      ["Processing_Latency_Days"],
}

Z_THRESHOLD = 3.0
IQR_MULTIPLIER = 1.5

df["Stat_Zscore_Anomaly"] = False
df["Stat_IQR_Anomaly"] = False
df["Stat_Anomaly_Fields"] = [[] for _ in range(len(df))]

field_level_results = []

for rtype, fields in FIELDS_BY_TYPE.items():
    type_mask = df["Record_Type"] == rtype
    subset = df.loc[type_mask]

    for field in fields:
        if field not in df.columns:
            continue

        values = subset[field]
        valid = values.dropna()
        if len(valid) < 10:
            continue  # not enough data to compute a meaningful baseline

        # ---- Z-score ----
        mean = valid.mean()
        std = valid.std()
        if std > 0:
            z_scores = (values - mean) / std
            z_flag = z_scores.abs() > Z_THRESHOLD
        else:
            z_flag = pd.Series(False, index=values.index)

        # ---- IQR ----
        q1, q3 = valid.quantile(0.25), valid.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - IQR_MULTIPLIER * iqr
        upper = q3 + IQR_MULTIPLIER * iqr
        if iqr > 0:
            iqr_flag = (values < lower) | (values > upper)
        else:
            iqr_flag = pd.Series(False, index=values.index)

        z_flag = z_flag.fillna(False)
        iqr_flag = iqr_flag.fillna(False)

        df.loc[type_mask, "Stat_Zscore_Anomaly"] |= z_flag
        df.loc[type_mask, "Stat_IQR_Anomaly"] |= iqr_flag

        combined_flag = z_flag | iqr_flag
        for idx in values.index[combined_flag]:
            df.at[idx, "Stat_Anomaly_Fields"] = df.at[idx, "Stat_Anomaly_Fields"] + [field]

        field_level_results.append({
            "record_type": rtype,
            "field": field,
            "mean": round(float(mean), 2),
            "std": round(float(std), 2),
            "iqr_lower_bound": round(float(lower), 2),
            "iqr_upper_bound": round(float(upper), 2),
            "zscore_flagged": int(z_flag.sum()),
            "iqr_flagged": int(iqr_flag.sum()),
            "combined_flagged": int(combined_flag.sum()),
        })

df["Stat_Is_Anomalous"] = df["Stat_Zscore_Anomaly"] | df["Stat_IQR_Anomaly"]

print(f"Z-score anomalies : {df['Stat_Zscore_Anomaly'].sum():,}")
print(f"IQR anomalies     : {df['Stat_IQR_Anomaly'].sum():,}")
print(f"Combined (either) : {df['Stat_Is_Anomalous'].sum():,} / {len(df):,} "
      f"({100*df['Stat_Is_Anomalous'].mean():.2f}%)")

print("\nPer-field breakdown:")
for r in field_level_results:
    print(f"  [{r['record_type']:<15}] {r['field']:<25} "
          f"z={r['zscore_flagged']:>4}  iqr={r['iqr_flagged']:>4}  combined={r['combined_flagged']:>4}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
findings = {
    "method": "Z-score (threshold=3.0) and IQR (multiplier=1.5), computed within each Record_Type group",
    "summary": {
        "zscore_anomalies": int(df["Stat_Zscore_Anomaly"].sum()),
        "iqr_anomalies": int(df["Stat_IQR_Anomaly"].sum()),
        "combined_anomalies": int(df["Stat_Is_Anomalous"].sum()),
        "combined_anomalies_pct": round(100 * df["Stat_Is_Anomalous"].mean(), 2),
    },
    "field_level_results": field_level_results,
}
with open(OUTPUT_PATH, "w") as f:
    json.dump(findings, f, indent=2, default=str)

print(f"\nSaved: {OUTPUT_PATH}")

Z-score anomalies : 300
IQR anomalies     : 735
Combined (either) : 735 / 10,000 (7.35%)

Per-field breakdown:
  [MEDICAL_CLAIM  ] Billed_Amount             z=   8  iqr= 341  combined= 341
  [MEDICAL_CLAIM  ] Allowed_Amount            z= 150  iqr= 311  combined= 311
  [MEDICAL_CLAIM  ] Paid_Amount               z= 149  iqr= 322  combined= 322
  [MEDICAL_CLAIM  ] Patient_Responsibility    z=   0  iqr=   0  combined=   0
  [MEDICAL_CLAIM  ] Processing_Latency_Days   z=   0  iqr=   0  combined=   0
  [PHARMACY_CLAIM ] Billed_Amount             z=  11  iqr= 179  combined= 179
  [PHARMACY_CLAIM ] Allowed_Amount            z=  48  iqr= 171  combined= 171
  [PHARMACY_CLAIM ] Paid_Amount               z=  47  iqr= 169  combined= 169
  [PHARMACY_CLAIM ] Patient_Responsibility    z=  65  iqr= 180  combined= 180
  [PHARMACY_CLAIM ] Days_Supply               z=   0  iqr=   0  combined=   0
  [PHARMACY_CLAIM ] Quantity_Dispensed        z=   0  iqr=   0  combined=   0
  [PHARMACY_CLAIM ] Processing_

In [ ]:
# ==============================================================================
# MODULE: Domain-Aware Isolation Forest Anomaly Detection Pipeline
# Assumes `df` is already present in memory.
# ==============================================================================

import json
import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

OUTPUT_DIR = "outputs"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "ml_isolation_forest_findings.json")

# ------------------------------------------------------------------------------
# STEP 0: Data Sanitization & Type Normalization
# ------------------------------------------------------------------------------
# Normalize key identifiers to strings (preserve alphanumeric IDs and leading zeros)
id_cols = ["Record_ID", "BENE_ID", "Provider_NPI", "Auth_Linked_ID"]
for col in id_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).replace({"nan": np.nan, "None": np.nan})

# Parse timestamp/date fields into proper datetime dtype
date_cols = [
    "Service_Date",
    "Service_End_Date",
    "Processed_Date",
    "Decision_Date",
    "Submission_Date",
]
for col in date_cols:
    if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"Loaded DataFrame: {len(df):,} rows x {len(df.columns)} columns")

# ------------------------------------------------------------------------------
# STEP 1: Feature Engineering (Contextual & Domain-Aware)
# ------------------------------------------------------------------------------
X = df.copy()

# Latency and Duration Metrics
if "Processing_Latency_Days" in X.columns:
    X["Days_To_Process"] = X["Processing_Latency_Days"]
elif "Processed_Date" in X.columns and "Submission_Date" in X.columns:
    X["Days_To_Process"] = (X["Processed_Date"] - X["Submission_Date"]).dt.days
else:
    X["Days_To_Process"] = np.nan

if "Decision_Date" in X.columns and "Processed_Date" in X.columns:
    X["Decision_Latency_Days"] = (
        X["Decision_Date"] - X["Processed_Date"]
    ).dt.days.fillna(0)
else:
    X["Decision_Latency_Days"] = 0

if "Service_End_Date" in X.columns and "Service_Date" in X.columns:
    X["Service_Duration_Days"] = (
        X["Service_End_Date"] - X["Service_Date"]
    ).dt.days
else:
    X["Service_Duration_Days"] = np.nan

# Financial Ratios (Replacing 0 denominators with NaN to avoid division-by-zero errors)
billed = (
    X["Billed_Amount"].replace(0, np.nan)
    if "Billed_Amount" in X.columns
    else np.nan
)
allowed = (
    X["Allowed_Amount"].replace(0, np.nan)
    if "Allowed_Amount" in X.columns
    else np.nan
)
paid = X["Paid_Amount"] if "Paid_Amount" in X.columns else np.nan

X["Paid_to_Billed_Ratio"] = (
    paid / billed if isinstance(paid, pd.Series) else np.nan
)
X["Allowed_to_Billed_Ratio"] = (
    allowed / billed if isinstance(allowed, pd.Series) else np.nan
)
X["Paid_to_Allowed_Ratio"] = (
    paid / allowed if isinstance(paid, pd.Series) else np.nan
)

# Operational Flags and Overruns
if (
    "Processing_Latency_Days" in X.columns
    and "SLA_Target_Days" in X.columns
):
    X["SLA_Overrun_Days"] = (
        X["Processing_Latency_Days"] - X["SLA_Target_Days"]
    )
else:
    X["SLA_Overrun_Days"] = 0

if "Status" in X.columns:
    X["Is_Denied_Or_Rejected"] = (
        X["Status"].isin(["DENIED", "REJECTED"]).astype(int)
    )
else:
    X["Is_Denied_Or_Rejected"] = 0

if "Retry_Count" in X.columns:
    X["Retry_Count_Bucket"] = (
        pd.cut(
            X["Retry_Count"].fillna(0),
            bins=[-np.inf, 0, 2, np.inf],
            labels=[0, 1, 2],
        )
        .astype(float)
        .fillna(0)
    )
else:
    X["Retry_Count_Bucket"] = 0.0

# Logical Business Missingness: Requires authorization but none is linked
auth_req = (
    X["Auth_Required_Flag"].fillna("N")
    if "Auth_Required_Flag" in X.columns
    else pd.Series("N", index=X.index)
)
rec_type = (
    X["Record_Type"].fillna("")
    if "Record_Type" in X.columns
    else pd.Series("", index=X.index)
)
auth_id = (
    X["Auth_Linked_ID"]
    if "Auth_Linked_ID" in X.columns
    else pd.Series(np.nan, index=X.index)
)

X["Unexpected_Missing_Count"] = (
    (auth_req == "Y") & (rec_type != "PRIOR_AUTH") & (auth_id.isna())
).astype(int)

# String-Safe Frequency Encodings (Preserves alphanumeric format without float coercion)
freq_targets = [
    ("Provider_NPI", "Provider_NPI_Frequency"),
    ("BENE_ID", "BENE_ID_Frequency"),
    ("Diagnosis_Code", "Diagnosis_Code_Frequency"),
    ("NDC_Code", "NDC_Code_Frequency"),
    ("Drug_Name", "Drug_Name_Frequency"),
]

for col, new_col in freq_targets:
    if col in X.columns:
        counts = X[col].dropna().value_counts()
        X[new_col] = X[col].map(counts).fillna(0)
    else:
        X[new_col] = 0

# ------------------------------------------------------------------------------
# STEP 2: Domain-Specific Partitioning & Feature Definition
# ------------------------------------------------------------------------------
DOMAIN_CONFIGS = {
    "MEDICAL_CLAIM": {
        "numeric": [
            "Billed_Amount",
            "Allowed_Amount",
            "Paid_Amount",
            "Patient_Responsibility",
            "Processing_Latency_Days",
            "Days_To_Process",
            "Decision_Latency_Days",
            "Service_Duration_Days",
            "Paid_to_Billed_Ratio",
            "Allowed_to_Billed_Ratio",
            "Paid_to_Allowed_Ratio",
            "SLA_Overrun_Days",
            "Is_Denied_Or_Rejected",
            "Retry_Count_Bucket",
            "Unexpected_Missing_Count",
            "Provider_NPI_Frequency",
            "BENE_ID_Frequency",
            "Diagnosis_Code_Frequency",
        ],
        "categorical": [
            "Provider_State",
            "Status",
            "Denial_Reason_Code",
            "Procedure_Code",
            "Source_System",
            "SLA_Breach_Flag",
            "Urgency_Flag",
            "Auth_Required_Flag",
        ],
    },
    "PHARMACY_CLAIM": {
        "numeric": [
            "Billed_Amount",
            "Allowed_Amount",
            "Paid_Amount",
            "Patient_Responsibility",
            "Days_Supply",
            "Quantity_Dispensed",
            "Processing_Latency_Days",
            "Days_To_Process",
            "Decision_Latency_Days",
            "Paid_to_Billed_Ratio",
            "Allowed_to_Billed_Ratio",
            "Paid_to_Allowed_Ratio",
            "SLA_Overrun_Days",
            "Is_Denied_Or_Rejected",
            "Retry_Count_Bucket",
            "Unexpected_Missing_Count",
            "Provider_NPI_Frequency",
            "BENE_ID_Frequency",
            "NDC_Code_Frequency",
            "Drug_Name_Frequency",
        ],
        "categorical": [
            "Provider_State",
            "Status",
            "Denial_Reason_Code",
            "Source_System",
            "SLA_Breach_Flag",
            "Urgency_Flag",
            "Auth_Required_Flag",
        ],
    },
    "PRIOR_AUTH": {
        "numeric": [
            "Processing_Latency_Days",
            "Days_To_Process",
            "Decision_Latency_Days",
            "Service_Duration_Days",
            "SLA_Overrun_Days",
            "Is_Denied_Or_Rejected",
            "Retry_Count_Bucket",
            "Provider_NPI_Frequency",
            "BENE_ID_Frequency",
            "Diagnosis_Code_Frequency",
        ],
        "categorical": [
            "Provider_State",
            "Status",
            "Denial_Reason_Code",
            "Procedure_Code",
            "Source_System",
            "SLA_Breach_Flag",
            "Urgency_Flag",
        ],
    },
}

# ------------------------------------------------------------------------------
# STEP 3: Model Training per Domain
# ------------------------------------------------------------------------------
# Initialize output target fields
df["ISO_Is_Anomaly"] = False
df["ISO_Raw_Score"] = np.nan
df["ISO_Severity_0to1"] = np.nan

fitted_feature_counts = {}

# Ensure Record_Type exists, default to ALL if missing
if "Record_Type" not in df.columns:
    df["Record_Type"] = "MEDICAL_CLAIM"

for domain_type, cfg in DOMAIN_CONFIGS.items():
    mask = df["Record_Type"] == domain_type
    subset_indices = df[mask].index

    if len(subset_indices) == 0:
        continue

    # Filter features available in the current dataset
    num_feats = [f for f in cfg["numeric"] if f in X.columns]
    cat_feats = [f for f in cfg["categorical"] if f in X.columns]

    # Preprocessing Pipeline
    # Using max_categories to prevent dimensionality explosion on high-cardinality codes
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(strategy="median"),
                        ),
                        ("scaler", StandardScaler()),
                    ]
                ),
                num_feats,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="most_frequent"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                                max_categories=30,
                            ),
                        ),
                    ]
                ),
                cat_feats,
            ),
        ]
    )

    domain_input = X.loc[subset_indices, num_feats + cat_feats]
    X_proc = preprocessor.fit_transform(domain_input)
    fitted_feature_counts[domain_type] = int(X_proc.shape[1])

    # Model instantiation with reproducibility and domain sensitivity
    iso = IsolationForest(
        n_estimators=200,
        contamination="auto",
        random_state=42,
        n_jobs=-1,
    )
    iso.fit(X_proc)

    # Scikit-learn outputs: -1 = Anomaly, 1 = Normal
    preds = iso.predict(X_proc)
    raw_scores = iso.decision_function(X_proc)

    # Convert raw score to standardized 0-1 severity (lower score = higher severity)
    score_min, score_max = raw_scores.min(), raw_scores.max()
    denom = (score_max - score_min) if (score_max - score_min) != 0 else 1e-9
    severity = np.clip((score_max - raw_scores) / denom, 0.0, 1.0)

    # Assign domain outputs back to main DataFrame
    df.loc[subset_indices, "ISO_Is_Anomaly"] = preds == -1
    df.loc[subset_indices, "ISO_Raw_Score"] = raw_scores
    df.loc[subset_indices, "ISO_Severity_0to1"] = severity

    print(
        f"Domain [{domain_type:<15}] Transformed Features: {X_proc.shape[1]:>3} | "
        f"Rows: {len(subset_indices):>6,} | Flagged Anomalies: {(preds == -1).sum():>5,} "
        f"({100 * (preds == -1).mean():.2f}%)"
    )

# ------------------------------------------------------------------------------
# STEP 4: Output Serialization & Artifact Generation
# ------------------------------------------------------------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

export_cols = [
    "Record_ID",
    "Record_Type",
    "BENE_ID",
    "Provider_NPI",
    "ISO_Is_Anomaly",
    "ISO_Raw_Score",
    "ISO_Severity_0to1",
]
available_export_cols = [c for c in export_cols if c in df.columns]

# Extract top 20 records with the highest anomaly severity score
top_20_anomalies = (
    df.sort_values("ISO_Severity_0to1", ascending=False)
    .head(20)[available_export_cols]
    .fillna("")
    .to_dict(orient="records")
)

total_records = len(df)
flagged_count = int(df["ISO_Is_Anomaly"].sum())
flagged_pct = round(100 * (flagged_count / max(total_records, 1)), 2)

findings = {
    "method": (
        "Domain-Partitioned IsolationForest(n_estimators=200, contamination='auto', random_state=42) "
        "with median/most-frequent imputation, standard scaling, and one-hot encoding (max_categories=30)."
    ),
    "features_per_domain": fitted_feature_counts,
    "summary": {
        "total_records_scanned": total_records,
        "isolation_forest_flagged_count": flagged_count,
        "isolation_forest_flagged_pct": flagged_pct,
    },
    "top_20_highest_severity_records": top_20_anomalies,
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(findings, f, indent=2, default=str)

print(f"\nSuccessfully generated outputs at: {OUTPUT_PATH}")
print(f"Total Anomalies Flagged: {flagged_count:,} / {total_records:,} ({flagged_pct}%)")

Loaded DataFrame: 10,000 rows x 65 columns


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Denial_Reason_Code' 'Procedure_Code' 'Urgency_Flag']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


Domain [MEDICAL_CLAIM  ] Transformed Features:  48 | Rows:  5,008 | Flagged Anomalies:    74 (1.48%)


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Urgency_Flag']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


Domain [PHARMACY_CLAIM ] Transformed Features:  43 | Rows:  2,992 | Flagged Anomalies:   388 (12.97%)


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Service_Duration_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Domain [PRIOR_AUTH     ] Transformed Features:  40 | Rows:  2,000 | Flagged Anomalies:   283 (14.15%)

Successfully generated outputs at: outputs/ml_isolation_forest_findings.json
Total Anomalies Flagged: 745 / 10,000 (7.45%)


In [ ]:
import os
import sqlite3
import json
import pandas as pd

# Ensure output directory exists
os.makedirs("outputs", exist_ok=True)
db_path = "outputs/audit_anomalies.db"

# 1. Filter only flagged anomalies sorted by severity
anomalies_df = df[df["ISO_Is_Anomaly"] == True].sort_values(
    "ISO_Severity_0to1", ascending=False
).copy()

# ------------------------------------------------------------
# FIX: Convert any list/dict columns (like Stat_Anomaly_Fields)
# into clean, SQL-compatible comma-separated strings or JSON
# ------------------------------------------------------------
for col in anomalies_df.columns:
    # Check if any cell in this column is a list or dict
    if anomalies_df[col].apply(lambda x: isinstance(x, (list, dict))).any():
        anomalies_df[col] = anomalies_df[col].apply(
            lambda x: ", ".join(map(str, x)) if isinstance(x, list)
            else (json.dumps(x) if isinstance(x, dict) else x)
        )

# 2. Connect to SQLite database
conn = sqlite3.connect(db_path)

# 3. Save master table of all flagged anomalies
anomalies_df.to_sql("all_anomalies", conn, if_exists="replace", index=False)

# 4. Save separate tables per domain for fast domain-specific queries
for domain in ["MEDICAL_CLAIM", "PHARMACY_CLAIM", "PRIOR_AUTH"]:
    domain_df = anomalies_df[anomalies_df["Record_Type"] == domain]
    if len(domain_df) > 0:
        table_name = f"anomalies_{domain.lower()}"
        domain_df.to_sql(table_name, conn, if_exists="replace", index=False)

# 5. Add B-Tree Indexes on key lookup columns
cursor = conn.cursor()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_rec_id ON all_anomalies(Record_ID);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_bene_id ON all_anomalies(BENE_ID);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_provider_npi ON all_anomalies(Provider_NPI);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_severity ON all_anomalies(ISO_Severity_0to1 DESC);")
conn.commit()

print(f"✅ SQLite database created successfully at: {db_path}")

# Verify stored tables and counts
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cursor.fetchall()]
print("\nStored Tables:")
for t in tables:
    count = cursor.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  • {t:<25}: {count:,} records")

conn.close()

✅ SQLite database created successfully at: outputs/audit_anomalies.db

Stored Tables:
  • all_anomalies            : 745 records
  • anomalies_medical_claim  : 74 records
  • anomalies_pharmacy_claim : 388 records
  • anomalies_prior_auth     : 283 records


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("outputs/audit_anomalies.db")

# List all tables
tables_df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
display(tables_df)

,name
0,all_anomalies
1,anomalies_medical_claim
2,anomalies_pharmacy_claim
3,anomalies_prior_auth


In [ ]:
query = """
SELECT
    Record_ID,
    Record_Type,
    BENE_ID,
    Provider_NPI,
    Billed_Amount,
    Paid_Amount,
    ISO_Severity_0to1,
    Stat_Anomaly_Fields
FROM all_anomalies
ORDER BY ISO_Severity_0to1 DESC
LIMIT 10;
"""

df_view = pd.read_sql_query(query, conn)
display(df_view)

conn.close()

,Record_ID,Record_Type,BENE_ID,Provider_NPI,Billed_Amount,Paid_Amount,ISO_Severity_0to1,Stat_Anomaly_Fields
0,MC101289,MEDICAL_CLAIM,-10000010257320.0,1801898556.0,16674.370000,13339.48,1.000000,"Billed_Amount, Allowed_Amount, Paid_Amount"
1,PH201432,PHARMACY_CLAIM,-10000010275879.0,1787233815.0,257151.103687,2122.63,1.000000,"Billed_Amount, Allowed_Amount, Paid_Amount"
2,PA300081,PRIOR_AUTH,-10000010256715.0,1502049183.0,NaN,NaN,1.000000,Processing_Latency_Days
3,MC100447,MEDICAL_CLAIM,-10000010257256.0,1821232869.0,11090.810000,11090.81,0.996699,"Billed_Amount, Allowed_Amount, Paid_Amount"
4,MC101667,MEDICAL_CLAIM,-10000010255239.0,1336255264.0,2178.500000,1742.81,0.962882,
5,PA301158_DUP,PRIOR_AUTH,-10000010259120.0,1845678368.0,NaN,NaN,0.954796,
6,PA301158,PRIOR_AUTH,-10000010259120.0,1845678368.0,NaN,NaN,0.954796,
7,PA300147,PRIOR_AUTH,-10000010260325.0,1216219100.0,NaN,NaN,0.951034,Processing_Latency_Days
8,PH202804,PHARMACY_CLAIM,-10000010255493.0,1912368008.0,2266.950000,1366.83,0.948917,"Billed_Amount, Patient_Responsibility"
9,PA301639_DUP,PRIOR_AUTH,-10000010273270.0,1993962280.0,NaN,NaN,0.948770,Processing_Latency_Days


In [ ]:
# ============================================================
# MODULE 3: Correlation & Relationship Analysis
# Runs after Module 2 in the same Colab session - uses `df`
# already in memory. Trains fresh, no files to upload.
#
# Checks whether known business relationships between fields
# still hold for each record. Unlike Module 1 (single-field
# outliers) or Module 2 (overall multivariate strangeness), this
# catches records where two RELATED fields disagree with each
# other - e.g. a paid amount that doesn't track the allowed
# amount the way it should, or a drug quantity that doesn't
# match the days-supply it was dispensed for.
# ============================================================

import pandas as pd
import numpy as np
import json
import os

from sklearn.linear_model import LinearRegression

OUTPUT_DIR = "outputs"
OUTPUT_PATH = f"{OUTPUT_DIR}/correlation_findings.json"

RESIDUAL_SIGMA_THRESHOLD = 3.0

# ------------------------------------------------------------
# RELATIONSHIP 1: Paid_Amount ~ Allowed_Amount
# Business expectation: paid amount should track the allowed
# amount closely (a plan pays close to, but usually at or below,
# what it allowed). A record that breaks this pattern signals a
# pricing/adjudication error or a manual override.
# ------------------------------------------------------------
corr_mask = df["Allowed_Amount"].notna() & df["Paid_Amount"].notna()

correlation_model = LinearRegression()
correlation_model.fit(df.loc[corr_mask, ["Allowed_Amount"]], df.loc[corr_mask, "Paid_Amount"])

pearson_r = df.loc[corr_mask, "Allowed_Amount"].corr(df.loc[corr_mask, "Paid_Amount"])

df["Correlation_Predicted_Paid"] = np.nan
df["Correlation_Residual"] = np.nan
df["Correlation_Anomaly"] = False

pred_paid = correlation_model.predict(df.loc[corr_mask, ["Allowed_Amount"]])
residual = df.loc[corr_mask, "Paid_Amount"].values - pred_paid
resid_std = residual.std()

df.loc[corr_mask, "Correlation_Predicted_Paid"] = pred_paid
df.loc[corr_mask, "Correlation_Residual"] = residual
df.loc[corr_mask, "Correlation_Anomaly"] = np.abs(residual) > RESIDUAL_SIGMA_THRESHOLD * resid_std

print("RELATIONSHIP 1: Paid_Amount ~ Allowed_Amount")
print(f"  Pearson r              : {pearson_r:.4f}")
print(f"  Fitted line             : Paid = {correlation_model.coef_[0]:.4f} * Allowed + {correlation_model.intercept_:.4f}")
print(f"  Residual std             : {resid_std:.2f}")
print(f"  Anomalies (|resid|>{RESIDUAL_SIGMA_THRESHOLD}sd): {df['Correlation_Anomaly'].sum():,}")

# ------------------------------------------------------------
# RELATIONSHIP 2: Quantity_Dispensed ~ Days_Supply
# Business expectation: the quantity dispensed should scale
# consistently with the days-supply prescribed (pharmacy claims
# only). A record that breaks this pattern signals a dispensing
# error or a mismatched drug/quantity/days-supply entry.
# ------------------------------------------------------------
qty_mask = (df["Record_Type"] == "PHARMACY_CLAIM") & df["Days_Supply"].notna() & df["Quantity_Dispensed"].notna()

quantity_supply_model = LinearRegression()
quantity_supply_model.fit(df.loc[qty_mask, ["Days_Supply"]], df.loc[qty_mask, "Quantity_Dispensed"])

qty_pearson_r = df.loc[qty_mask, "Days_Supply"].corr(df.loc[qty_mask, "Quantity_Dispensed"])

df["Quantity_Supply_Predicted"] = np.nan
df["Quantity_Supply_Residual"] = np.nan
df["Quantity_Supply_Anomaly"] = False

pred_qty = quantity_supply_model.predict(df.loc[qty_mask, ["Days_Supply"]])
qresidual = df.loc[qty_mask, "Quantity_Dispensed"].values - pred_qty
qresid_std = qresidual.std()

df.loc[qty_mask, "Quantity_Supply_Predicted"] = pred_qty
df.loc[qty_mask, "Quantity_Supply_Residual"] = qresidual
df.loc[qty_mask, "Quantity_Supply_Anomaly"] = np.abs(qresidual) > RESIDUAL_SIGMA_THRESHOLD * qresid_std

print()
print("RELATIONSHIP 2: Quantity_Dispensed ~ Days_Supply")
print(f"  Pearson r               : {qty_pearson_r:.4f}")
print(f"  Fitted line              : Quantity = {quantity_supply_model.coef_[0]:.4f} * Days_Supply + {quantity_supply_model.intercept_:.4f}")
print(f"  Residual std              : {qresid_std:.2f}")
print(f"  Anomalies (|resid|>{RESIDUAL_SIGMA_THRESHOLD}sd): {df['Quantity_Supply_Anomaly'].sum():,}")

# ------------------------------------------------------------
# Combine into one Module-3 verdict per record
# ------------------------------------------------------------
df["Correlation_Module_Anomaly_Count"] = (
    df["Correlation_Anomaly"].astype(int) + df["Quantity_Supply_Anomaly"].astype(int)
)
df["Correlation_Module_Is_Anomalous"] = df["Correlation_Module_Anomaly_Count"] > 0

print()
print(f"Combined Module 3 anomalies (either relationship broken): "
      f"{df['Correlation_Module_Is_Anomalous'].sum():,} / {len(df):,} "
      f"({100*df['Correlation_Module_Is_Anomalous'].mean():.2f}%)")

# ------------------------------------------------------------
# Save findings
# ------------------------------------------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

record_cols = ["Record_ID", "Record_Type", "BENE_ID",
               "Correlation_Anomaly", "Correlation_Residual",
               "Quantity_Supply_Anomaly", "Quantity_Supply_Residual",
               "Correlation_Module_Anomaly_Count", "Correlation_Module_Is_Anomalous"]

findings = {
    "relationships_checked": [
        {
            "name": "Paid_Amount ~ Allowed_Amount",
            "pearson_r": round(float(pearson_r), 4),
            "fitted_coefficient": round(float(correlation_model.coef_[0]), 4),
            "fitted_intercept": round(float(correlation_model.intercept_), 4),
            "residual_std": round(float(resid_std), 2),
            "anomalies_flagged": int(df["Correlation_Anomaly"].sum()),
        },
        {
            "name": "Quantity_Dispensed ~ Days_Supply",
            "pearson_r": round(float(qty_pearson_r), 4),
            "fitted_coefficient": round(float(quantity_supply_model.coef_[0]), 4),
            "fitted_intercept": round(float(quantity_supply_model.intercept_), 4),
            "residual_std": round(float(qresid_std), 2),
            "anomalies_flagged": int(df["Quantity_Supply_Anomaly"].sum()),
        },
    ],
    "summary": {
        "combined_anomalies": int(df["Correlation_Module_Is_Anomalous"].sum()),
        "combined_anomalies_pct": round(100 * df["Correlation_Module_Is_Anomalous"].mean(), 2),
    },
    "sample_flagged_records": (
        df[df["Correlation_Module_Is_Anomalous"]][record_cols]
          .head(20)
          .fillna("")
          .to_dict(orient="records")
    ),
}
with open(OUTPUT_PATH, "w") as f:
    json.dump(findings, f, indent=2, default=str)

print(f"\nSaved: {OUTPUT_PATH}")

RELATIONSHIP 1: Paid_Amount ~ Allowed_Amount
  Pearson r              : 0.9856
  Fitted line             : Paid = 0.9960 * Allowed + 15.3589
  Residual std             : 357.62
  Anomalies (|resid|>3.0sd): 71

RELATIONSHIP 2: Quantity_Dispensed ~ Days_Supply
  Pearson r               : 0.9287
  Fitted line              : Quantity = 1.1550 * Days_Supply + -0.2097
  Residual std              : 10.38
  Anomalies (|resid|>3.0sd): 9

Combined Module 3 anomalies (either relationship broken): 80 / 10,000 (0.80%)

Saved: outputs/correlation_findings.json


In [ ]:
import os
import sqlite3
import json
import pandas as pd

# ------------------------------------------------------------
# 1. Prepare & Sanitize Anomaly Data from in-memory `df`
# ------------------------------------------------------------
# Filter only flagged anomaly records
anomalies_df = df[df["ISO_Is_Anomaly"] == True].sort_values(
    "ISO_Severity_0to1", ascending=False
).copy()

# Fix: Convert Python lists/dicts (e.g., Stat_Anomaly_Fields) into strings
# so SQLite can store them without throwing a ProgrammingError
for col in anomalies_df.columns:
    if anomalies_df[col].apply(lambda x: isinstance(x, (list, dict))).any():
        anomalies_df[col] = anomalies_df[col].apply(
            lambda x: ", ".join(map(str, x)) if isinstance(x, list)
            else (json.dumps(x) if isinstance(x, dict) else x)
        )

# ------------------------------------------------------------
# 2. Insert into SQLite Database
# ------------------------------------------------------------
os.makedirs("outputs", exist_ok=True)
db_path = "outputs/audit_anomalies.db"

conn = sqlite3.connect(db_path)

# Write master anomaly table
anomalies_df.to_sql("all_anomalies", conn, if_exists="replace", index=False)

# Optional: Add index on severity for fast sorting
cursor = conn.cursor()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_severity ON all_anomalies(ISO_Severity_0to1 DESC);")
conn.commit()

print(f"Successfully inserted {len(anomalies_df):,} anomaly records into {db_path}\n")

# ------------------------------------------------------------
# 3. Execute Your Query & Display Top 10 Anomalies
# ------------------------------------------------------------
query = """
SELECT
    Record_ID,
    Record_Type,
    BENE_ID,
    Provider_NPI,
    Billed_Amount,
    Paid_Amount,
    ISO_Severity_0to1,
    Stat_Anomaly_Fields
FROM all_anomalies
ORDER BY ISO_Severity_0to1 DESC
LIMIT 10;
"""

df_view = pd.read_sql_query(query, conn)
conn.close()

# Display the interactive/tabular result
display(df_view)

Successfully inserted 745 anomaly records into outputs/audit_anomalies.db



,Record_ID,Record_Type,BENE_ID,Provider_NPI,Billed_Amount,Paid_Amount,ISO_Severity_0to1,Stat_Anomaly_Fields
0,MC101289,MEDICAL_CLAIM,-10000010257320.0,1801898556.0,16674.370000,13339.48,1.000000,"Billed_Amount, Allowed_Amount, Paid_Amount"
1,PH201432,PHARMACY_CLAIM,-10000010275879.0,1787233815.0,257151.103687,2122.63,1.000000,"Billed_Amount, Allowed_Amount, Paid_Amount"
2,PA300081,PRIOR_AUTH,-10000010256715.0,1502049183.0,NaN,NaN,1.000000,Processing_Latency_Days
3,MC100447,MEDICAL_CLAIM,-10000010257256.0,1821232869.0,11090.810000,11090.81,0.996699,"Billed_Amount, Allowed_Amount, Paid_Amount"
4,MC101667,MEDICAL_CLAIM,-10000010255239.0,1336255264.0,2178.500000,1742.81,0.962882,
5,PA301158_DUP,PRIOR_AUTH,-10000010259120.0,1845678368.0,NaN,NaN,0.954796,
6,PA301158,PRIOR_AUTH,-10000010259120.0,1845678368.0,NaN,NaN,0.954796,
7,PA300147,PRIOR_AUTH,-10000010260325.0,1216219100.0,NaN,NaN,0.951034,Processing_Latency_Days
8,PH202804,PHARMACY_CLAIM,-10000010255493.0,1912368008.0,2266.950000,1366.83,0.948917,"Billed_Amount, Patient_Responsibility"
9,PA301639_DUP,PRIOR_AUTH,-10000010273270.0,1993962280.0,NaN,NaN,0.948770,Processing_Latency_Days
